# EpiShield AI — ETL Pipeline

## 1. Project Overview

This notebook implements the ETL (Extract, Transform, Load) pipeline for the EpiShield AI project.

The pipeline processes four prepared datasets: Population, Regional, National, and Notes.

The objective is to extract the prepared data, define source-to-target mappings, apply the required transformations, validate data integrity, and load the final outputs.

The initial output destination will be CSV files, followed by integration with Microsoft SQL Server.

## 2. ETL Objectives

- Extract the prepared datasets.
- Analyze source schemas and data structures.
- Define source-to-target data mappings.
- Design the SQL Server target schema.
- Apply documented transformation rules.
- Integrate datasets using validated relationships.
- Perform data validation.
- Load the final outputs.
- Maintain execution logs and error handling.

In [3]:
# Import required libraries

import pandas as pd
import numpy as np
import os
import logging

from pathlib import Path
from datetime import datetime

## 3. Project Configuration

This section defines the project directories and central configuration variables used throughout the ETL pipeline.

Centralized configuration improves maintainability and allows the pipeline to switch between CSV storage and SQL Server in later stages.

In [5]:
# Define project directories

PROJECT_ROOT = Path.cwd().parent.parent

SRC_DIR = PROJECT_ROOT / "Src"

DATA_DIR = SRC_DIR / "Data"
PROCESSED_DIR = SRC_DIR / "Processed Data"

ETL_OUTPUT_DIR = SRC_DIR / "ETL Output"
ETL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Source Data:", DATA_DIR)
print("Processed Data:", PROCESSED_DIR)
print("ETL Output:", ETL_OUTPUT_DIR)

Project Root: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1
Source Data: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\Data
Processed Data: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\Processed Data
ETL Output: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output


In [6]:
print("Source Data Exists:", DATA_DIR.exists())
print("Processed Data Exists:", PROCESSED_DIR.exists())

print("\nProcessed Files:")

for file in PROCESSED_DIR.glob("*"):
    print(file.name)

Source Data Exists: True
Processed Data Exists: True

Processed Files:
national_clean.csv
notes_clean.csv
population_clean.csv
regional_clean.csv


## 4. Extract Phase

### Objective
Load the four preprocessed datasets from the Processed Data directory into separate Pandas DataFrames.

### Source Datasets
- Population Dataset
- Regional Dataset
- National Dataset
- Notes Dataset

### Extraction Strategy
- Read the prepared CSV files.
- Preserve identifier columns as strings.
- Preserve missing values without replacing them with zeros.
- Keep each dataset separate during extraction.
- Validate the extracted dataset dimensions.

In [7]:
# Extract prepared datasets

population_df = pd.read_csv(
    PROCESSED_DIR / "population_clean.csv"
)

regional_df = pd.read_csv(
    PROCESSED_DIR / "regional_clean.csv"
)

national_df = pd.read_csv(
    PROCESSED_DIR / "national_clean.csv"
)

notes_df = pd.read_csv(
    PROCESSED_DIR / "notes_clean.csv"
)

print("All datasets extracted successfully.")

All datasets extracted successfully.


In [8]:
# Validate extracted dataset dimensions

datasets = {
    "Population": population_df,
    "Regional": regional_df,
    "National": national_df,
    "Notes": notes_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

Population: 210 rows, 12 columns
Regional: 37401 rows, 30 columns
National: 1781 rows, 24 columns
Notes: 1617 rows, 2 columns


## 5. Source Schema Analysis

### Objective
Analyze the extracted datasets to understand their structures, column names, data types, and missing values before defining the target schema.

### Analysis Scope
- Dataset dimensions.
- Column names and data types.
- Missing value counts.
- Identifier and date fields.
- Potential primary and foreign keys.
- Dataset granularity (grain).

In [9]:
for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"{name} Dataset")
    print(f"{'='*50}")

    print("Shape:", df.shape)
    print("\nData Types:")
    print(df.dtypes)


Population Dataset
Shape: (210, 12)

Data Types:
codice_regione               int64
codice_nuts_1               object
descrizione_nuts_1          object
codice_nuts_2               object
denominazione_regione       object
sigla_regione               object
latitudine_regione         float64
longitudine_regione        float64
range_eta                   object
totale_genere_maschile       int64
totale_genere_femminile      int64
totale_generale              int64
dtype: object

Regional Dataset
Shape: (37401, 30)

Data Types:
data                                       object
stato                                      object
codice_regione                              int64
denominazione_regione                      object
lat                                       float64
long                                      float64
ricoverati_con_sintomi                      int64
terapia_intensiva                           int64
totale_ospedalizzati                        int64
isolamento_domic

In [10]:
for name, df in datasets.items():
    print(f"\n--- {name} Missing Values ---")

    missing = df.isna().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("No missing values detected.")
    else:
        print(missing)


--- Population Missing Values ---
No missing values detected.

--- Regional Missing Values ---
casi_da_sospetto_diagnostico              33999
casi_da_screening                         33999
casi_testati                               1155
note                                      27562
ingressi_terapia_intensiva                 5943
note_test                                 37204
note_casi                                 32767
totale_positivi_test_molecolare            6846
totale_positivi_test_antigenico_rapido     6846
tamponi_test_molecolare                    6846
tamponi_test_antigenico_rapido             6846
codice_nuts_1                              6846
codice_nuts_2                              6846
dtype: int64

--- National Missing Values ---
casi_da_sospetto_diagnostico              1619
casi_da_screening                         1619
casi_testati                                55
note                                      1737
ingressi_terapia_intensiva                 283

## 6. Dataset Grain and Key Identification

### Objective
Identify the natural grain and candidate keys of each dataset before defining source-to-target mappings.

### Validation Scope
- Check candidate key uniqueness.
- Identify duplicate key combinations.
- Verify the expected number of records per key.
- Distinguish business keys from technical identifiers.
- Document relationships supported by the source data.

### Important Considerations
No primary key or foreign key will be assigned without validating its uniqueness and referential integrity.

In [11]:
# Population: Region + Age Range

print("Population Key Check:")
print(
    "Duplicate Region-Age combinations:",
    population_df.duplicated(
        subset=["codice_regione", "range_eta"]
    ).sum()
)


# Regional: Region + Date

print("\nRegional Key Check:")
print(
    "Duplicate Region-Date combinations:",
    regional_df.duplicated(
        subset=["codice_regione", "data"]
    ).sum()
)


# National: Date

print("\nNational Key Check:")
print(
    "Duplicate Dates:",
    national_df.duplicated(subset=["data"]).sum()
)


# Notes: Date

print("\nNotes Key Check:")
print(
    "Duplicate Dates:",
    notes_df.duplicated(subset=["data"]).sum()
)

Population Key Check:
Duplicate Region-Age combinations: 0

Regional Key Check:
Duplicate Region-Date combinations: 0

National Key Check:
Duplicate Dates: 0

Notes Key Check:
Duplicate Dates: 0


### Key Validation Results

All candidate keys passed the duplicate validation test.

| Dataset | Candidate Key | Duplicate Records |
|---|---|---:|
| Population | codice_regione + range_eta | 0 |
| Regional | codice_regione + data | 0 |
| National | data | 0 |
| Notes | data | 0 |

The candidate keys are unique within their respective datasets.

These keys are considered candidate business keys. Final Primary Key
and Foreign Key constraints will be defined after validating null values,
data types, and referential integrity.

In [12]:
key_columns = {
    "Population": (population_df, ["codice_regione", "range_eta"]),
    "Regional": (regional_df, ["codice_regione", "data"]),
    "National": (national_df, ["data"]),
    "Notes": (notes_df, ["data"])
}

for name, (df, columns) in key_columns.items():
    print(f"\n{name} Key Null Check:")

    for col in columns:
        print(f"{col}: {df[col].isna().sum()} missing values")


Population Key Null Check:
codice_regione: 0 missing values
range_eta: 0 missing values

Regional Key Null Check:
codice_regione: 0 missing values
data: 0 missing values

National Key Null Check:
data: 0 missing values

Notes Key Null Check:
data: 0 missing values


### Candidate Key Nullability Validation

All columns participating in the candidate keys were checked for
missing values.

No missing values were found in any candidate key column.

Combined with the uniqueness checks, all four candidate keys passed
the initial validation.

Final database constraints will be determined during target schema design.

In [13]:
population_regions = set(population_df["codice_regione"].dropna())
regional_regions = set(regional_df["codice_regione"].dropna())

regional_regions_not_in_population = (
    regional_regions - population_regions
)

print("Regional unique regions:", len(regional_regions))
print("Population unique regions:", len(population_regions))

print(
    "Regional regions not found in Population:",
    len(regional_regions_not_in_population)
)

print(
    "Unmatched region codes:",
    sorted(regional_regions_not_in_population)
)

Regional unique regions: 21
Population unique regions: 21
Regional regions not found in Population: 0
Unmatched region codes: []


In [14]:
population_regions_not_in_regional = (
    population_regions - regional_regions
)

print(
    "Population regions not found in Regional:",
    len(population_regions_not_in_regional)
)

print(
    "Unmatched region codes:",
    sorted(population_regions_not_in_regional)
)

Population regions not found in Regional: 0
Unmatched region codes: []


### Population–Regional Referential Integrity

The region codes were compared in both directions.

- All 21 unique region codes in Regional exist in Population.
- All 21 unique region codes in Population exist in Regional.
- No unmatched region codes were found.

Therefore, both datasets contain the same set of 21 regions.

A one-to-many relationship is identified conceptually:
- Population: multiple age-range records per region.
- Regional: multiple daily records per region.

The relationship requires a separate Region lookup table
or another appropriate schema design before defining SQL Server
Primary Key and Foreign Key constraints.

In [15]:
national_dates = set(national_df["data"].dropna())
regional_dates = set(regional_df["data"].dropna())

national_dates_not_in_regional = national_dates - regional_dates
regional_dates_not_in_national = regional_dates - national_dates

print("National unique dates:", len(national_dates))
print("Regional unique dates:", len(regional_dates))

print(
    "National dates not found in Regional:",
    len(national_dates_not_in_regional)
)

print(
    "Regional dates not found in National:",
    len(regional_dates_not_in_national)
)

print(
    "National unmatched dates:",
    sorted(national_dates_not_in_regional)[:10]
)

print(
    "Regional unmatched dates:",
    sorted(regional_dates_not_in_national)[:10]
)

National unique dates: 1781
Regional unique dates: 1781
National dates not found in Regional: 5
Regional dates not found in National: 5
National unmatched dates: ['2020-02-29 18:00:00', '2020-03-01 18:00:00', '2020-03-04 18:00:00', '2020-03-05 18:00:00', '2020-03-06 18:00:00']
Regional unmatched dates: ['2020-02-29 17:00:00', '2020-03-01 17:00:00', '2020-03-04 17:00:00', '2020-03-05 17:00:00', '2020-03-06 17:00:00']


In [16]:
# Compare dates at the calendar-day level
national_calendar_dates = pd.to_datetime(
    national_df["data"]
).dt.date

regional_calendar_dates = pd.to_datetime(
    regional_df["data"]
).dt.date

national_days = set(national_calendar_dates)
regional_days = set(regional_calendar_dates)

print("National unique calendar days:", len(national_days))
print("Regional unique calendar days:", len(regional_days))

print(
    "National days not found in Regional:",
    len(national_days - regional_days)
)

print(
    "Regional days not found in National:",
    len(regional_days - national_days)
)

National unique calendar days: 1781
Regional unique calendar days: 1781
National days not found in Regional: 0
Regional days not found in National: 0


### National–Regional Date Referential Integrity

The initial timestamp comparison identified five unmatched timestamps
in each direction.

Further investigation showed that these timestamps share the same
calendar dates but have different time components:

- National records: 18:00:00
- Regional records: 17:00:00

After comparing calendar dates only:

- National unique calendar days: 1,781
- Regional unique calendar days: 1,781
- National-only calendar days: 0
- Regional-only calendar days: 0

Conclusion:
Both datasets cover exactly the same calendar days.

The timestamp differences must be handled through an explicit
date-standardization rule before establishing database relationships.
No source values were modified during validation.

In [17]:
notes_days = set(pd.to_datetime(notes_df["data"]).dt.date)

print("Notes unique calendar days:", len(notes_days))

print(
    "Notes days not found in National:",
    len(notes_days - national_days)
)

print(
    "Notes days not found in Regional:",
    len(notes_days - regional_days)
)

print(
    "National days without Notes:",
    len(national_days - notes_days)
)

print(
    "Regional days without Notes:",
    len(regional_days - notes_days)
)

Notes unique calendar days: 1617
Notes days not found in National: 0
Notes days not found in Regional: 0
National days without Notes: 164
Regional days without Notes: 164


### Notes–National–Regional Date Coverage

All 1,617 unique calendar days in Notes exist in both National
and Regional datasets.

However:
- 164 National calendar days have no corresponding Notes record.
- 164 Regional calendar days have no corresponding Notes record.

Notes therefore covers 1,617 out of 1,781 calendar days (90.79%).

Important:
The absence of a Notes record does not imply the absence of an event
or observation.

Calendar-date matching alone does not establish a valid semantic
relationship between Notes and Regional records.

No automatic merge or data modification was performed.

In [18]:
population_regions = (
    population_df[
        [
            "codice_regione",
            "denominazione_regione",
            "sigla_regione",
            "codice_nuts_1",
            "codice_nuts_2"
        ]
    ]
    .drop_duplicates()
    .sort_values("codice_regione")
)

regional_regions = (
    regional_df[
        ["codice_regione", "denominazione_regione"]
    ]
    .drop_duplicates()
    .sort_values("codice_regione")
)

print("Population Regions:")
display(population_regions)

print("\nRegional Regions:")
display(regional_regions)

Population Regions:


,codice_regione,denominazione_regione,sigla_regione,codice_nuts_1,codice_nuts_2
120,1,Piemonte,PIE,ITC,ITC1
190,2,Valle d'Aosta,VDA,ITC,ITC2
90,3,Lombardia,LOM,ITC,ITC4
200,5,Veneto,VEN,ITH,ITH3
60,6,Friuli Venezia Giulia,FVG,ITH,ITH4
80,7,Liguria,LIG,ITC,ITC3
50,8,Emilia-Romagna,EMR,ITH,ITH5
160,9,Toscana,TOS,ITI,ITI1
180,10,Umbria,UMB,ITI,ITI2
100,11,Marche,MAR,ITI,ITI3



Regional Regions:


,codice_regione,denominazione_regione
13,1,Piemonte
19,2,Valle d'Aosta
8,3,Lombardia
20,5,Veneto
5,6,Friuli Venezia Giulia
7,7,Liguria
4,8,Emilia-Romagna
17,9,Toscana
18,10,Umbria
9,11,Marche


In [19]:
# Get one record per region from Population
population_region_names = (
    population_df[
        ["codice_regione", "denominazione_regione"]
    ]
    .drop_duplicates()
)

# Get one record per region from Regional
regional_region_names = (
    regional_df[
        ["codice_regione", "denominazione_regione"]
    ]
    .drop_duplicates()
)

# Compare both datasets
region_comparison = population_region_names.merge(
    regional_region_names,
    on="codice_regione",
    how="outer",
    suffixes=("_population", "_regional"),
    indicator=True,
    validate="one_to_one"
)

# Identify different region names
region_comparison["name_match"] = (
    region_comparison["denominazione_regione_population"]
    == region_comparison["denominazione_regione_regional"]
)

display(
    region_comparison[
        [
            "codice_regione",
            "denominazione_regione_population",
            "denominazione_regione_regional",
            "_merge",
            "name_match"
        ]
    ]
)

print(
    "Different region names:",
    (~region_comparison["name_match"]).sum()
)

,codice_regione,denominazione_regione_population,denominazione_regione_regional,_merge,name_match
0,1,Piemonte,Piemonte,both,True
1,2,Valle d'Aosta,Valle d'Aosta,both,True
2,3,Lombardia,Lombardia,both,True
3,5,Veneto,Veneto,both,True
4,6,Friuli Venezia Giulia,Friuli Venezia Giulia,both,True
5,7,Liguria,Liguria,both,True
6,8,Emilia-Romagna,Emilia-Romagna,both,True
7,9,Toscana,Toscana,both,True
8,10,Umbria,Umbria,both,True
9,11,Marche,Marche,both,True


Different region names: 2


In [20]:
print("Population region codes:")
print(sorted(population_df["codice_regione"].unique()))

print("\nRegional region codes:")
print(sorted(regional_df["codice_regione"].unique()))

Population region codes:
[np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]

Regional region codes:
[np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]


In [21]:
population_codes_check = pd.read_csv(
    PROCESSED_DIR / "population_clean.csv",
    dtype={"codice_regione": "string"}
)

regional_codes_check = pd.read_csv(
    PROCESSED_DIR / "regional_clean.csv",
    dtype={"codice_regione": "string"}
)

print("Population codes:")
print(sorted(population_codes_check["codice_regione"].unique()))

print("\nRegional codes:")
print(sorted(regional_codes_check["codice_regione"].unique()))

Population codes:
['01', '02', '03', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22']

Regional codes:
['01', '02', '03', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22']


## 7. Target Data Modeling

### 7.1 DimRegion Construction

In [22]:
# Create DimRegion from Population dataset

dim_region = (
    population_codes_check[
        [
            "codice_regione",
            "denominazione_regione",
            "sigla_regione",
            "codice_nuts_1",
            "codice_nuts_2"
        ]
    ]
    .drop_duplicates()
    .sort_values("codice_regione")
    .reset_index(drop=True)
)

# Rename source columns to target naming convention

dim_region = dim_region.rename(columns={
    "codice_regione": "region_code",
    "denominazione_regione": "region_name",
    "sigla_regione": "region_abbreviation",
    "codice_nuts_1": "nuts_1_code",
    "codice_nuts_2": "nuts_2_code"
})

display(dim_region)

print("DimRegion Shape:", dim_region.shape)
print("Duplicate Region Codes:", dim_region["region_code"].duplicated().sum())
print("Missing Values:")
print(dim_region.isna().sum())

,region_code,region_name,region_abbreviation,nuts_1_code,nuts_2_code
0,01,Piemonte,PIE,ITC,ITC1
1,02,Valle d'Aosta,VDA,ITC,ITC2
2,03,Lombardia,LOM,ITC,ITC4
3,05,Veneto,VEN,ITH,ITH3
4,06,Friuli Venezia Giulia,FVG,ITH,ITH4
5,07,Liguria,LIG,ITC,ITC3
6,08,Emilia-Romagna,EMR,ITH,ITH5
7,09,Toscana,TOS,ITI,ITI1
8,10,Umbria,UMB,ITI,ITI2
9,11,Marche,MAR,ITI,ITI3


DimRegion Shape: (21, 5)
Duplicate Region Codes: 0
Missing Values:
region_code            0
region_name            0
region_abbreviation    0
nuts_1_code            0
nuts_2_code            0
dtype: int64


In [23]:
# Create DimDate from National and Regional datasets

# Extract calendar dates from both datasets
national_dates = pd.to_datetime(
    national_df["data"]
).dt.normalize()

regional_dates = pd.to_datetime(
    regional_df["data"]
).dt.normalize()

# Combine dates from both datasets
all_dates = pd.concat(
    [national_dates, regional_dates],
    ignore_index=True
)

# Create unique sorted calendar dates
dim_date = pd.DataFrame({
    "full_date": sorted(all_dates.dropna().unique())
})

# Generate date attributes
dim_date["year"] = dim_date["full_date"].dt.year

dim_date["quarter"] = dim_date["full_date"].dt.quarter

dim_date["month"] = dim_date["full_date"].dt.month

dim_date["month_name"] = dim_date["full_date"].dt.month_name()

dim_date["day"] = dim_date["full_date"].dt.day

dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1

# Generate date key in YYYYMMDD format
dim_date["date_key"] = (
    dim_date["full_date"].dt.strftime("%Y%m%d").astype("int64")
)

# Arrange columns
dim_date = dim_date[
    [
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week"
    ]
]

# Display results
display(dim_date)

# Validation checks
print("DimDate Shape:", dim_date.shape)

print(
    "Duplicate Date Keys:",
    dim_date["date_key"].duplicated().sum()
)

print("\nMissing Values:")
print(dim_date.isna().sum())

print(
    "\nDate Range:",
    dim_date["full_date"].min(),
    "to",
    dim_date["full_date"].max()
)

,date_key,full_date,year,quarter,month,month_name,day,day_of_week
0,20200224,2020-02-24,2020,1,2,February,24,1
1,20200225,2020-02-25,2020,1,2,February,25,2
2,20200226,2020-02-26,2020,1,2,February,26,3
3,20200227,2020-02-27,2020,1,2,February,27,4
4,20200228,2020-02-28,2020,1,2,February,28,5
...,...,...,...,...,...,...,...,...
1776,20250104,2025-01-04,2025,1,1,January,4,6
1777,20250105,2025-01-05,2025,1,1,January,5,7
1778,20250106,2025-01-06,2025,1,1,January,6,1
1779,20250107,2025-01-07,2025,1,1,January,7,2


DimDate Shape: (1781, 8)
Duplicate Date Keys: 0

Missing Values:
date_key       0
full_date      0
year           0
quarter        0
month          0
month_name     0
day            0
day_of_week    0
dtype: int64

Date Range: 2020-02-24 00:00:00 to 2025-01-08 00:00:00


#### 7.2.1 DimDate Validation Results

The Date Dimension was successfully constructed using the National and Regional datasets.

**Validation Results:**
- Total records: 1,781.
- Total attributes: 8.
- Duplicate date keys: 0.
- Missing values: 0.
- Start date: 2020-02-24.
- End date: 2025-01-08.

**Conclusion**

The Date Dimension passed all initial validation checks and provides a standardized calendar reference for the analytical data model.

Original source timestamps remain unchanged.

In [24]:
# Inspect age groups in Population dataset

print("Unique Age Groups:")
print(
    sorted(
        population_codes_check["range_eta"].dropna().unique()
    )
)

print("\nAge Group Counts:")
print(
    population_codes_check["range_eta"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nMissing Age Groups:",
    population_codes_check["range_eta"].isna().sum()
)

Unique Age Groups:
['0-15', '05-11', '12-19', '16-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']

Age Group Counts:
range_eta
0-15     20
05-11     1
12-19     1
16-19    20
20-29    21
30-39    21
40-49    21
50-59    21
60-69    21
70-79    21
80-89    21
90+      21
Name: count, dtype: int64

Missing Age Groups: 0


In [25]:
# Analyze age groups by region

age_region_distribution = (
    population_codes_check
    .groupby(
        ["denominazione_regione", "range_eta"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["denominazione_regione", "range_eta"]
    )
)

display(age_region_distribution)

print(
    "Total Region-Age combinations:",
    len(age_region_distribution)
)

print(
    "Unique Regions:",
    age_region_distribution["denominazione_regione"].nunique()
)

print(
    "Unique Age Groups:",
    age_region_distribution["range_eta"].nunique()
)

,denominazione_regione,range_eta,record_count
0,Abruzzo,0-15,1
1,Abruzzo,16-19,1
2,Abruzzo,20-29,1
3,Abruzzo,30-39,1
4,Abruzzo,40-49,1
...,...,...,...
205,Veneto,50-59,1
206,Veneto,60-69,1
207,Veneto,70-79,1
208,Veneto,80-89,1


Total Region-Age combinations: 210
Unique Regions: 21
Unique Age Groups: 12


In [26]:
# Identify regions using the exceptional age groups

exceptional_age_groups = ["05-11", "12-19"]

exceptional_records = population_codes_check[
    population_codes_check["range_eta"].isin(
        exceptional_age_groups
    )
][
    [
        "codice_regione",
        "denominazione_regione",
        "range_eta",
        "totale_generale"
    ]
].sort_values(["codice_regione", "range_eta"])

display(exceptional_records)

print(
    "Regions using exceptional age groups:",
    exceptional_records["denominazione_regione"].unique()
)

,codice_regione,denominazione_regione,range_eta,totale_generale
90,03,Lombardia,05-11,646260
91,03,Lombardia,12-19,770162


Regions using exceptional age groups: ['Lombardia']


In [27]:
# Create DimAgeGroup from Population dataset

dim_age_group = (
    population_codes_check[["range_eta"]]
    .drop_duplicates()
    .sort_values("range_eta")
    .reset_index(drop=True)
)

# Generate surrogate key
dim_age_group.insert(
    0,
    "age_group_key",
    range(1, len(dim_age_group) + 1)
)

# Rename column
dim_age_group = dim_age_group.rename(
    columns={
        "range_eta": "age_group_label"
    }
)

# Display results
display(dim_age_group)

# Validation checks
print("DimAgeGroup Shape:", dim_age_group.shape)

print(
    "Duplicate Age Group Keys:",
    dim_age_group["age_group_key"].duplicated().sum()
)

print(
    "Duplicate Age Group Labels:",
    dim_age_group["age_group_label"].duplicated().sum()
)

print("\nMissing Values:")
print(dim_age_group.isna().sum())

,age_group_key,age_group_label
0,1,0-15
1,2,05-11
2,3,12-19
3,4,16-19
4,5,20-29
5,6,30-39
6,7,40-49
7,8,50-59
8,9,60-69
9,10,70-79


DimAgeGroup Shape: (12, 2)
Duplicate Age Group Keys: 0
Duplicate Age Group Labels: 0

Missing Values:
age_group_key      0
age_group_label    0
dtype: int64


In [28]:
# Create FactPopulation from Population dataset

fact_population = population_codes_check[
    [
        "codice_regione",
        "range_eta",
        "totale_genere_maschile",
        "totale_genere_femminile",
        "totale_generale"
    ]
].copy()

# Map Age Group Key
fact_population = fact_population.merge(
    dim_age_group,
    left_on="range_eta",
    right_on="age_group_label",
    how="left",
    validate="many_to_one"
)

# Rename columns
fact_population = fact_population.rename(columns={
    "codice_regione": "region_code",
    "totale_genere_maschile": "male_population",
    "totale_genere_femminile": "female_population",
    "totale_generale": "total_population"
})

# Keep target columns
fact_population = fact_population[
    [
        "region_code",
        "age_group_key",
        "male_population",
        "female_population",
        "total_population"
    ]
]

# Display results
display(fact_population)

print("FactPopulation Shape:", fact_population.shape)

print("\nMissing Values:")
print(fact_population.isna().sum())

print(
    "\nDuplicate Region-Age Combinations:",
    fact_population.duplicated(
        subset=["region_code", "age_group_key"]
    ).sum()
)

,region_code,age_group_key,male_population,female_population,total_population
0,13,1,87423,82474,169897
1,13,4,24402,22349,46751
2,13,5,67472,62611,130083
3,13,6,76472,74241,150713
4,13,7,94745,95315,190060
...,...,...,...,...,...
205,05,8,399024,400436,799460
206,05,9,293238,309972,603210
207,05,10,228669,265774,494443
208,05,11,115439,177591,293030


FactPopulation Shape: (210, 5)

Missing Values:
region_code          0
age_group_key        0
male_population      0
female_population    0
total_population     0
dtype: int64

Duplicate Region-Age Combinations: 0


In [29]:
# 1. Validate population totals

fact_population["calculated_total"] = (
    fact_population["male_population"]
    + fact_population["female_population"]
)

total_mismatches = fact_population[
    fact_population["calculated_total"]
    != fact_population["total_population"]
]

print("Population Total Mismatches:", len(total_mismatches))


# 2. Validate Region Keys

unmatched_regions = fact_population[
    ~fact_population["region_code"].isin(
        dim_region["region_code"]
    )
]

print("Unmatched Region Codes:", len(unmatched_regions))


# 3. Validate Age Group Keys

unmatched_age_groups = fact_population[
    ~fact_population["age_group_key"].isin(
        dim_age_group["age_group_key"]
    )
]

print("Unmatched Age Group Keys:", len(unmatched_age_groups))


# 4. Display mismatches if found

if len(total_mismatches) > 0:
    print("\nPopulation Total Mismatches Details:")
    display(total_mismatches)

if len(unmatched_regions) > 0:
    print("\nUnmatched Regions Details:")
    display(unmatched_regions)

if len(unmatched_age_groups) > 0:
    print("\nUnmatched Age Groups Details:")
    display(unmatched_age_groups)

Population Total Mismatches: 0
Unmatched Region Codes: 0
Unmatched Age Group Keys: 0


#### 7.4.2 FactPopulation Validation Results

The Population Fact Table passed all validation checks.

**Validation Results:**
- Total records: 210.
- Population total mismatches: 0.
- Unmatched region codes: 0.
- Unmatched age-group keys: 0.
- Missing values: 0.
- Duplicate Region-Age combinations: 0.

**Conclusion**

The numerical consistency and dimensional key integrity of FactPopulation were successfully verified.

No automatic data corrections were required.

In [30]:
print("National Dataset Columns:")
print(national_df.columns.tolist())

print("\nNational Dataset Shape:")
print(national_df.shape)

print("\nNational Dataset Data Types:")
print(national_df.dtypes)

National Dataset Columns:
['data', 'stato', 'ricoverati_con_sintomi', 'terapia_intensiva', 'totale_ospedalizzati', 'isolamento_domiciliare', 'totale_positivi', 'variazione_totale_positivi', 'nuovi_positivi', 'dimessi_guariti', 'deceduti', 'casi_da_sospetto_diagnostico', 'casi_da_screening', 'totale_casi', 'tamponi', 'casi_testati', 'note', 'ingressi_terapia_intensiva', 'note_test', 'note_casi', 'totale_positivi_test_molecolare', 'totale_positivi_test_antigenico_rapido', 'tamponi_test_molecolare', 'tamponi_test_antigenico_rapido']

National Dataset Shape:
(1781, 24)

National Dataset Data Types:
data                                       object
stato                                      object
ricoverati_con_sintomi                      int64
terapia_intensiva                           int64
totale_ospedalizzati                        int64
isolamento_domiciliare                      int64
totale_positivi                             int64
variazione_totale_positivi                  int6

In [31]:
# Define National Fact measures

national_measure_mapping = {
    "ricoverati_con_sintomi": "symptomatic_hospitalized",
    "terapia_intensiva": "intensive_care",
    "totale_ospedalizzati": "total_hospitalized",
    "isolamento_domiciliare": "home_isolation",
    "totale_positivi": "total_positive",
    "variazione_totale_positivi": "positive_variation",
    "nuovi_positivi": "new_positive_cases",
    "dimessi_guariti": "recovered",
    "deceduti": "deaths",
    "casi_da_sospetto_diagnostico": "suspected_diagnosis_cases",
    "casi_da_screening": "screening_cases",
    "totale_casi": "total_cases",
    "tamponi": "total_tests",
    "casi_testati": "tested_cases",
    "ingressi_terapia_intensiva": "intensive_care_admissions",
    "totale_positivi_test_molecolare": "molecular_positive_cases",
    "totale_positivi_test_antigenico_rapido": "rapid_antigen_positive_cases",
    "tamponi_test_molecolare": "molecular_tests",
    "tamponi_test_antigenico_rapido": "rapid_antigen_tests"
}

# Select date and numerical measures
fact_national = national_df[
    ["data"] + list(national_measure_mapping.keys())
].copy()

# Extract calendar date
fact_national["full_date"] = (
    pd.to_datetime(fact_national["data"]).dt.normalize()
)

# Map Date Key
fact_national = fact_national.merge(
    dim_date[["full_date", "date_key"]],
    on="full_date",
    how="left",
    validate="one_to_one"
)

# Rename numerical measures
fact_national = fact_national.rename(
    columns=national_measure_mapping
)

# Select target columns
fact_national = fact_national[
    ["date_key"] + list(national_measure_mapping.values())
]

# Display results
display(fact_national)

print("FactNational Shape:", fact_national.shape)

print("\nMissing Values:")
print(fact_national.isna().sum())

print(
    "\nDuplicate Date Keys:",
    fact_national["date_key"].duplicated().sum()
)

print(
    "\nUnmatched Date Keys:",
    fact_national["date_key"].isna().sum()
)

,date_key,symptomatic_hospitalized,intensive_care,total_hospitalized,home_isolation,total_positive,positive_variation,new_positive_cases,recovered,deaths,suspected_diagnosis_cases,screening_cases,total_cases,total_tests,tested_cases,intensive_care_admissions,molecular_positive_cases,rapid_antigen_positive_cases,molecular_tests,rapid_antigen_tests
0,20200224,101,26,127,94,221,0,221,1,7,NaN,NaN,229,4324,NaN,NaN,NaN,NaN,NaN,NaN
1,20200225,114,35,150,162,311,90,93,1,10,NaN,NaN,322,8623,NaN,NaN,NaN,NaN,NaN,NaN
2,20200226,128,36,164,221,385,74,78,3,12,NaN,NaN,400,9587,NaN,NaN,NaN,NaN,NaN,NaN
3,20200227,248,56,304,284,588,203,250,45,17,NaN,NaN,650,12014,NaN,NaN,NaN,NaN,NaN,NaN
4,20200228,345,64,409,412,821,233,238,46,21,NaN,NaN,888,15695,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1776,20250104,1278,45,1323,201663,202986,74,280,26562161,198666,NaN,NaN,26963813,284325249,70571636.0,3.0,9973077.0,16990736.0,103275200.0,181050049.0
1777,20250105,1267,48,1315,201806,203121,135,196,26562218,198670,NaN,NaN,26964009,284329947,70572388.0,3.0,9973129.0,16990880.0,103276127.0,181053820.0
1778,20250106,1289,49,1338,201783,203121,0,101,26562319,198670,NaN,NaN,26964110,284333343,70573017.0,2.0,9973158.0,16990952.0,103276857.0,181056486.0
1779,20250107,1314,48,1362,201844,203206,85,155,26562384,198674,NaN,NaN,26964264,284338302,70573883.0,0.0,9973220.0,16991044.0,103277626.0,181060676.0


FactNational Shape: (1781, 20)

Missing Values:
date_key                           0
symptomatic_hospitalized           0
intensive_care                     0
total_hospitalized                 0
home_isolation                     0
total_positive                     0
positive_variation                 0
new_positive_cases                 0
recovered                          0
deaths                             0
suspected_diagnosis_cases       1619
screening_cases                 1619
total_cases                        0
total_tests                        0
tested_cases                      55
intensive_care_admissions        283
molecular_positive_cases         326
rapid_antigen_positive_cases     326
molecular_tests                  326
rapid_antigen_tests              326
dtype: int64

Duplicate Date Keys: 0

Unmatched Date Keys: 0


#### 7.5.2 FactNational Data Preservation Validation

**Objective**

Verify that the transformation process preserved the original National dataset records and numerical measurements.

**Validation Scope**

- Compare source and target record counts.
- Verify date key uniqueness.
- Compare numerical values before and after transformation.
- Verify that missing-value counts remain unchanged.
- Confirm that no source measurements were modified.

**Expected Results**

- 1,781 source and target records.
- No duplicated date keys.
- No unmatched records.
- No changes in numerical values.
- Missing values preserved without replacement.

**Important**

Missing values must remain NULL and must not be replaced with zero.

In [32]:
# Prepare source data for comparison

source_national_check = national_df[
    ["data"] + list(national_measure_mapping.keys())
].copy()

source_national_check["full_date"] = (
    pd.to_datetime(source_national_check["data"]).dt.normalize()
)

source_national_check = source_national_check.merge(
    dim_date[["full_date", "date_key"]],
    on="full_date",
    how="left",
    validate="one_to_one"
)

source_national_check = source_national_check.rename(
    columns=national_measure_mapping
)

# Compare source and target using date_key

comparison = source_national_check[
    ["date_key"] + list(national_measure_mapping.values())
].merge(
    fact_national,
    on="date_key",
    how="outer",
    suffixes=("_source", "_target"),
    indicator=True,
    validate="one_to_one"
)

# Check record preservation

print("Source Records:", len(source_national_check))
print("Target Records:", len(fact_national))
print("Unmatched Records:", (comparison["_merge"] != "both").sum())

# Check numerical values and missing values

for col in national_measure_mapping.values():

    source_values = comparison[f"{col}_source"]
    target_values = comparison[f"{col}_target"]

    mismatch = ~(
        source_values.eq(target_values)
        | (source_values.isna() & target_values.isna())
    )

    print(
        f"{col}: Value Mismatches = {mismatch.sum()}"
    )

Source Records: 1781
Target Records: 1781
Unmatched Records: 0
symptomatic_hospitalized: Value Mismatches = 0
intensive_care: Value Mismatches = 0
total_hospitalized: Value Mismatches = 0
home_isolation: Value Mismatches = 0
total_positive: Value Mismatches = 0
positive_variation: Value Mismatches = 0
new_positive_cases: Value Mismatches = 0
recovered: Value Mismatches = 0
deaths: Value Mismatches = 0
suspected_diagnosis_cases: Value Mismatches = 0
screening_cases: Value Mismatches = 0
total_cases: Value Mismatches = 0
total_tests: Value Mismatches = 0
tested_cases: Value Mismatches = 0
intensive_care_admissions: Value Mismatches = 0
molecular_positive_cases: Value Mismatches = 0
rapid_antigen_positive_cases: Value Mismatches = 0
molecular_tests: Value Mismatches = 0
rapid_antigen_tests: Value Mismatches = 0


In [33]:
print("Regional Dataset Shape:")
print(regional_df.shape)

print("\nRegional Dataset Columns:")
print(regional_df.columns.tolist())

print("\nRegional Dataset Data Types:")
print(regional_df.dtypes)

Regional Dataset Shape:
(37401, 30)

Regional Dataset Columns:
['data', 'stato', 'codice_regione', 'denominazione_regione', 'lat', 'long', 'ricoverati_con_sintomi', 'terapia_intensiva', 'totale_ospedalizzati', 'isolamento_domiciliare', 'totale_positivi', 'variazione_totale_positivi', 'nuovi_positivi', 'dimessi_guariti', 'deceduti', 'casi_da_sospetto_diagnostico', 'casi_da_screening', 'totale_casi', 'tamponi', 'casi_testati', 'note', 'ingressi_terapia_intensiva', 'note_test', 'note_casi', 'totale_positivi_test_molecolare', 'totale_positivi_test_antigenico_rapido', 'tamponi_test_molecolare', 'tamponi_test_antigenico_rapido', 'codice_nuts_1', 'codice_nuts_2']

Regional Dataset Data Types:
data                                       object
stato                                      object
codice_regione                              int64
denominazione_regione                      object
lat                                       float64
long                                      float64
ricov

In [34]:
# Define Regional Fact measures

regional_measure_mapping = {
    "ricoverati_con_sintomi": "symptomatic_hospitalized",
    "terapia_intensiva": "intensive_care",
    "totale_ospedalizzati": "total_hospitalized",
    "isolamento_domiciliare": "home_isolation",
    "totale_positivi": "total_positive",
    "variazione_totale_positivi": "positive_variation",
    "nuovi_positivi": "new_positive_cases",
    "dimessi_guariti": "recovered",
    "deceduti": "deaths",
    "casi_da_sospetto_diagnostico": "suspected_diagnosis_cases",
    "casi_da_screening": "screening_cases",
    "totale_casi": "total_cases",
    "tamponi": "total_tests",
    "casi_testati": "tested_cases",
    "ingressi_terapia_intensiva": "intensive_care_admissions",
    "totale_positivi_test_molecolare": "molecular_positive_cases",
    "totale_positivi_test_antigenico_rapido": "rapid_antigen_positive_cases",
    "tamponi_test_molecolare": "molecular_tests",
    "tamponi_test_antigenico_rapido": "rapid_antigen_tests"
}

# Select source columns
fact_regional = regional_codes_check[
    ["data", "codice_regione"]
    + list(regional_measure_mapping.keys())
].copy()

# Standardize calendar date
fact_regional["full_date"] = (
    pd.to_datetime(fact_regional["data"]).dt.normalize()
)

# Map Region Dimension
fact_regional = fact_regional.merge(
    dim_region[["region_code"]],
    left_on="codice_regione",
    right_on="region_code",
    how="left",
    validate="many_to_one"
)

# Map Date Dimension
fact_regional = fact_regional.merge(
    dim_date[["full_date", "date_key"]],
    on="full_date",
    how="left",
    validate="many_to_one"
)

# Rename numerical measures
fact_regional = fact_regional.rename(
    columns=regional_measure_mapping
)

# Select target columns
fact_regional = fact_regional[
    ["region_code", "date_key"]
    + list(regional_measure_mapping.values())
]

# Display results
display(fact_regional)

# Initial structural validation
print("FactRegional Shape:", fact_regional.shape)

print("\nMissing Values:")
print(fact_regional.isna().sum())

print(
    "\nDuplicate Region-Date Combinations:",
    fact_regional.duplicated(
        subset=["region_code", "date_key"]
    ).sum()
)

print(
    "\nUnmatched Region Keys:",
    fact_regional["region_code"].isna().sum()
)

print(
    "Unmatched Date Keys:",
    fact_regional["date_key"].isna().sum()
)

,region_code,date_key,symptomatic_hospitalized,intensive_care,total_hospitalized,home_isolation,total_positive,positive_variation,new_positive_cases,recovered,...,suspected_diagnosis_cases,screening_cases,total_cases,total_tests,tested_cases,intensive_care_admissions,molecular_positive_cases,rapid_antigen_positive_cases,molecular_tests,rapid_antigen_tests
0,13,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,5,NaN,NaN,NaN,NaN,NaN,NaN
1,17,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,18,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,15,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,10,NaN,NaN,NaN,NaN,NaN,NaN
4,08,20200224,10,2,12,6,18,0,18,0,...,NaN,NaN,18,148,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37396,19,20250108,93,3,96,1415,1511,2,2,1822259,...,NaN,NaN,1836915,16989378,11325135.0,0.0,543730.0,1293185.0,5517733.0,11471645.0
37397,09,20250108,100,2,102,497,599,12,56,1656260,...,NaN,NaN,1669583,17608804,5503836.0,0.0,739065.0,930518.0,7176939.0,10431865.0
37398,10,20250108,16,0,16,666,682,0,0,459245,...,NaN,NaN,462464,5213580,834988.0,0.0,161000.0,301464.0,1766601.0,3446979.0
37399,02,20250108,0,0,0,1,1,-2,0,52191,...,NaN,NaN,52784,614028,149916.0,0.0,16498.0,36286.0,146786.0,467242.0


FactRegional Shape: (37401, 21)

Missing Values:
region_code                         0
date_key                            0
symptomatic_hospitalized            0
intensive_care                      0
total_hospitalized                  0
home_isolation                      0
total_positive                      0
positive_variation                  0
new_positive_cases                  0
recovered                           0
deaths                              0
suspected_diagnosis_cases       33999
screening_cases                 33999
total_cases                         0
total_tests                         0
tested_cases                     1155
intensive_care_admissions        5943
molecular_positive_cases         6846
rapid_antigen_positive_cases     6846
molecular_tests                  6846
rapid_antigen_tests              6846
dtype: int64

Duplicate Region-Date Combinations: 0

Unmatched Region Keys: 0
Unmatched Date Keys: 0


In [35]:
# Prepare source Regional data for comparison

source_regional_check = regional_codes_check[
    ["data", "codice_regione"]
    + list(regional_measure_mapping.keys())
].copy()

# Standardize calendar date
source_regional_check["full_date"] = (
    pd.to_datetime(source_regional_check["data"]).dt.normalize()
)

# Map Date Key
source_regional_check = source_regional_check.merge(
    dim_date[["full_date", "date_key"]],
    on="full_date",
    how="left",
    validate="many_to_one"
)

# Rename columns
source_regional_check = source_regional_check.rename(
    columns=regional_measure_mapping
)

# Rename region code
source_regional_check = source_regional_check.rename(
    columns={"codice_regione": "region_code"}
)

# Select comparison columns
comparison_columns = list(regional_measure_mapping.values())

# Compare source and target
regional_comparison = source_regional_check[
    ["region_code", "date_key"] + comparison_columns
].merge(
    fact_regional,
    on=["region_code", "date_key"],
    how="outer",
    suffixes=("_source", "_target"),
    indicator=True,
    validate="one_to_one"
)

# Record validation
print("Source Records:", len(source_regional_check))
print("Target Records:", len(fact_regional))

print(
    "Unmatched Records:",
    (regional_comparison["_merge"] != "both").sum()
)

# Compare all numerical measures
print("\nNumerical Value Validation:")

for col in comparison_columns:

    source_values = regional_comparison[f"{col}_source"]
    target_values = regional_comparison[f"{col}_target"]

    mismatch = ~(
        source_values.eq(target_values)
        | (source_values.isna() & target_values.isna())
    )

    print(f"{col}: Value Mismatches = {mismatch.sum()}")

Source Records: 37401
Target Records: 37401
Unmatched Records: 0

Numerical Value Validation:
symptomatic_hospitalized: Value Mismatches = 0
intensive_care: Value Mismatches = 0
total_hospitalized: Value Mismatches = 0
home_isolation: Value Mismatches = 0
total_positive: Value Mismatches = 0
positive_variation: Value Mismatches = 0
new_positive_cases: Value Mismatches = 0
recovered: Value Mismatches = 0
deaths: Value Mismatches = 0
suspected_diagnosis_cases: Value Mismatches = 0
screening_cases: Value Mismatches = 0
total_cases: Value Mismatches = 0
total_tests: Value Mismatches = 0
tested_cases: Value Mismatches = 0
intensive_care_admissions: Value Mismatches = 0
molecular_positive_cases: Value Mismatches = 0
rapid_antigen_positive_cases: Value Mismatches = 0
molecular_tests: Value Mismatches = 0
rapid_antigen_tests: Value Mismatches = 0


### 7.7 Notes Data Modeling and Analysis

**Objective**

Analyze the Notes dataset to determine its role and appropriate representation in the target data model.

**Source Dataset:** `notes_clean.csv`

**Description**

The Notes dataset contains dated textual observations associated with the source data.

**Analysis Scope**
- Inspect the dataset structure.
- Review the date and note attributes.
- Identify missing and duplicated notes.
- Examine the relationship between Notes and the National and Regional datasets.
- Determine whether notes represent national observations, regional observations, or mixed textual information.

**Transformation Considerations**
- Preserve the original Italian text.
- Preserve the original source dates.
- Do not associate notes with regions based on date alone.
- Do not discard missing or repeated notes without an approved rule.

**Expected Outcome**

A documented decision regarding the appropriate target structure for Notes.

In [36]:
# Inspect Notes dataset

print("Notes Shape:", notes_df.shape)

print("\nNotes Columns:")
print(notes_df.columns.tolist())

print("\nData Types:")
print(notes_df.dtypes)

print("\nMissing Values:")
print(notes_df.isna().sum())

print("\nDuplicate Rows:")
print(notes_df.duplicated().sum())

print("\nUnique Notes:")
print(notes_df["note"].nunique())

print("\nSample Notes:")
display(notes_df[["data", "note"]].head(10))

Notes Shape: (1617, 2)

Notes Columns:
['data', 'note']

Data Types:
data    object
note    object
dtype: object

Missing Values:
data     0
note    14
dtype: int64

Duplicate Rows:
0

Unique Notes:
1593

Sample Notes:


,data,note
0,2020-03-07 17:00:00,La Provincia di Brescia ha comunicato dati par...
1,2020-03-10 17:00:00,La Regione Lombardia ha comunicato dati parziali
2,2020-06-01 17:00:00,[altre date in fase di caricamento]
3,2020-06-25 17:00:00,La Regione Marche ha effettuato un ricalcolo s...
4,2020-06-26 17:00:00,La Regione Emilia Romagna ha effettuato un ric...
5,2020-07-01 17:00:00,La Regione Marche ha comunicato il seguente ri...
6,2020-07-06 17:00:00,La Regione Friuli Venezia Giulia riporta una d...
7,2020-07-07 17:00:00,La Regione Puglia riporta un caso totale in me...
8,2020-07-08 17:00:00,La Regione Lazio segnala che al totale dei cas...
9,2020-07-09 17:00:00,"Le Regioni Liguria, Piemonte e Valle d'Aosta r..."


In [37]:
# Explore region references in Notes

region_names = (
    dim_region["region_name"]
    .dropna()
    .unique()
)

notes_analysis = notes_df[["data", "note"]].copy()

notes_analysis["mentioned_regions"] = (
    notes_analysis["note"]
    .apply(
        lambda note: [
            region
            for region in region_names
            if pd.notna(note)
            and region.lower() in note.lower()
        ]
    )
)

notes_analysis["region_count"] = (
    notes_analysis["mentioned_regions"].str.len()
)

print("Notes with no region references:",
      (notes_analysis["region_count"] == 0).sum())

print("Notes mentioning one region:",
      (notes_analysis["region_count"] == 1).sum())

print("Notes mentioning multiple regions:",
      (notes_analysis["region_count"] > 1).sum())

print("\nSample notes mentioning multiple regions:")

display(
    notes_analysis[
        notes_analysis["region_count"] > 1
    ][["data", "note", "mentioned_regions"]].head(10)
)

Notes with no region references: 75
Notes mentioning one region: 152
Notes mentioning multiple regions: 1390

Sample notes mentioning multiple regions:


,data,note,mentioned_regions
5,2020-07-01 17:00:00,La Regione Marche ha comunicato il seguente ri...,"[Marche, Puglia]"
6,2020-07-06 17:00:00,La Regione Friuli Venezia Giulia riporta una d...,"[Piemonte, Friuli Venezia Giulia]"
7,2020-07-07 17:00:00,La Regione Puglia riporta un caso totale in me...,"[Lazio, Puglia]"
9,2020-07-09 17:00:00,"Le Regioni Liguria, Piemonte e Valle d'Aosta r...","[Piemonte, Valle d'Aosta, Liguria]"
12,2020-07-15 17:00:00,La regione Basilicata ha comunicato un ricalco...,"[Basilicata, Calabria]"
14,2020-07-19 17:00:00,La Regione Molise segnala che 1 caso del 18 lu...,"[Molise, Basilicata]"
16,2020-07-22 17:00:00,La regione Abruzzo comunica il seguente ricalc...,"[Abruzzo, Basilicata]"
18,2020-07-26 17:00:00,La Regione Abruzzo comunica il seguente ricalc...,"[Abruzzo, Basilicata]"
19,2020-07-27 17:00:00,L'ABRUZZO segnala che i dati di ieri (domenica...,"[Abruzzo, Sardegna]"
20,2020-07-28 17:00:00,La Regione BASILICATA segnala che il notevole ...,"[Basilicata, Calabria, Sardegna, Bolzano]"


In [38]:
# Prepare Notes dates

notes_date_check = notes_df.copy()

notes_date_check["full_date"] = (
    pd.to_datetime(notes_date_check["data"]).dt.normalize()
)

# Match with DimDate

notes_date_check = notes_date_check.merge(
    dim_date[["full_date", "date_key"]],
    on="full_date",
    how="left",
    validate="many_to_one"
)

# Validation results

print(
    "Total Notes Records:",
    len(notes_date_check)
)

print(
    "Unmatched Date Keys:",
    notes_date_check["date_key"].isna().sum()
)

print(
    "Duplicate Calendar Dates:",
    notes_date_check["full_date"].duplicated().sum()
)

print(
    "Unique Calendar Dates:",
    notes_date_check["full_date"].nunique()
)

print(
    "Notes per Date Distribution:"
)

print(
    notes_date_check.groupby("full_date").size()
    .value_counts()
    .sort_index()
)

Total Notes Records: 1617
Unmatched Date Keys: 0
Duplicate Calendar Dates: 0
Unique Calendar Dates: 1617
Notes per Date Distribution:
1    1617
Name: count, dtype: int64


In [39]:
# Create DimNote from Notes dataset

dim_note = notes_date_check[
    ["date_key", "note"]
].copy()

# Generate surrogate key
dim_note.insert(
    0,
    "note_key",
    range(1, len(dim_note) + 1)
)

# Rename columns
dim_note = dim_note.rename(
    columns={
        "note": "note_text"
    }
)

# Display results
display(dim_note)

# Structural validation
print("DimNote Shape:", dim_note.shape)

print(
    "\nDuplicate Note Keys:",
    dim_note["note_key"].duplicated().sum()
)

print(
    "Missing Note Keys:",
    dim_note["note_key"].isna().sum()
)

print(
    "Missing Date Keys:",
    dim_note["date_key"].isna().sum()
)

print(
    "Missing Note Texts:",
    dim_note["note_text"].isna().sum()
)

print(
    "Unique Note Texts:",
    dim_note["note_text"].nunique()
)

,note_key,date_key,note_text
0,1,20200307,La Provincia di Brescia ha comunicato dati par...
1,2,20200310,La Regione Lombardia ha comunicato dati parziali
2,3,20200601,[altre date in fase di caricamento]
3,4,20200625,La Regione Marche ha effettuato un ricalcolo s...
4,5,20200626,La Regione Emilia Romagna ha effettuato un ric...
...,...,...,...
1612,1613,20250104,La Regione Basilicata comunica che il dato rel...
1613,1614,20250105,La Regione Basilicata comunica che il dato rel...
1614,1615,20250106,La regione Basilicata comunica che il dato rel...
1615,1616,20250107,La Regione Basilicata comunica che il dato rel...


DimNote Shape: (1617, 3)

Duplicate Note Keys: 0
Missing Note Keys: 0
Missing Date Keys: 0
Missing Note Texts: 14
Unique Note Texts: 1593


### 7.8 Target Data Model Integration and Relationship Validation

**Objective**

Review the constructed Dimension and Fact Tables and validate their structural relationships before database loading.

**Tables Included**

Dimension Tables:
- DimRegion
- DimDate
- DimAgeGroup
- DimNote

Fact Tables:
- FactPopulation
- FactNational
- FactRegional

**Validation Scope**
- Verify primary key uniqueness.
- Verify foreign key references.
- Confirm Fact Table grain.
- Review table relationships.
- Identify missing or duplicated references.
- Document the proposed relational schema.

**Important**

This stage validates the DataFrame-level model. Actual SQL Server constraints will be created and tested during the database loading phase.

In [40]:
# Primary Key Validation

primary_keys = {
    "DimRegion": (dim_region, "region_code"),
    "DimDate": (dim_date, "date_key"),
    "DimAgeGroup": (dim_age_group, "age_group_key"),
    "DimNote": (dim_note, "note_key")
}

for table_name, (df, key) in primary_keys.items():

    print(f"\n{table_name}")
    print("Duplicate Keys:", df[key].duplicated().sum())
    print("Missing Keys:", df[key].isna().sum())
    print("Unique Keys:", df[key].nunique())


DimRegion
Duplicate Keys: 0
Missing Keys: 0
Unique Keys: 21

DimDate
Duplicate Keys: 0
Missing Keys: 0
Unique Keys: 1781

DimAgeGroup
Duplicate Keys: 0
Missing Keys: 0
Unique Keys: 12

DimNote
Duplicate Keys: 0
Missing Keys: 0
Unique Keys: 1617


In [41]:
# Foreign Key Validation

foreign_key_checks = [
    ("FactPopulation", fact_population, "region_code",
     "DimRegion", dim_region, "region_code"),

    ("FactPopulation", fact_population, "age_group_key",
     "DimAgeGroup", dim_age_group, "age_group_key"),

    ("FactNational", fact_national, "date_key",
     "DimDate", dim_date, "date_key"),

    ("FactRegional", fact_regional, "region_code",
     "DimRegion", dim_region, "region_code"),

    ("FactRegional", fact_regional, "date_key",
     "DimDate", dim_date, "date_key"),

    ("DimNote", dim_note, "date_key",
     "DimDate", dim_date, "date_key")
]

for source_name, source_df, source_col, target_name, target_df, target_col in foreign_key_checks:

    unmatched = ~source_df[source_col].isin(target_df[target_col])

    print(f"\n{source_name} -> {target_name}")
    print("Null Foreign Keys:", source_df[source_col].isna().sum())
    print("Unmatched Foreign Keys:", unmatched.sum())


FactPopulation -> DimRegion
Null Foreign Keys: 0
Unmatched Foreign Keys: 0

FactPopulation -> DimAgeGroup
Null Foreign Keys: 0
Unmatched Foreign Keys: 0

FactNational -> DimDate
Null Foreign Keys: 0
Unmatched Foreign Keys: 0

FactRegional -> DimRegion
Null Foreign Keys: 0
Unmatched Foreign Keys: 0

FactRegional -> DimDate
Null Foreign Keys: 0
Unmatched Foreign Keys: 0

DimNote -> DimDate
Null Foreign Keys: 0
Unmatched Foreign Keys: 0


In [42]:
# Fact Table Grain Validation

grain_checks = {
    "FactPopulation": {
        "df": fact_population,
        "keys": ["region_code", "age_group_key"],
        "expected_rows": 210
    },

    "FactNational": {
        "df": fact_national,
        "keys": ["date_key"],
        "expected_rows": 1781
    },

    "FactRegional": {
        "df": fact_regional,
        "keys": ["region_code", "date_key"],
        "expected_rows": 37401
    }
}

for table_name, config in grain_checks.items():

    df = config["df"]
    keys = config["keys"]

    duplicate_count = df.duplicated(subset=keys).sum()

    print(f"\n{table_name}")
    print("Expected Rows:", config["expected_rows"])
    print("Actual Rows:", len(df))
    print("Duplicate Grain Combinations:", duplicate_count)
    print("Grain Validation:", 
          "PASSED" if (
              len(df) == config["expected_rows"]
              and duplicate_count == 0
          ) else "FAILED")


FactPopulation
Expected Rows: 210
Actual Rows: 210
Duplicate Grain Combinations: 0
Grain Validation: PASSED

FactNational
Expected Rows: 1781
Actual Rows: 1781
Duplicate Grain Combinations: 0
Grain Validation: PASSED

FactRegional
Expected Rows: 37401
Actual Rows: 37401
Duplicate Grain Combinations: 0
Grain Validation: PASSED


In [43]:
# Final Data Model Inventory

target_tables = {
    "DimRegion": dim_region,
    "DimDate": dim_date,
    "DimAgeGroup": dim_age_group,
    "DimNote": dim_note,
    "FactPopulation": fact_population,
    "FactNational": fact_national,
    "FactRegional": fact_regional
}

for table_name, df in target_tables.items():

    print(f"\n{'=' * 50}")
    print(f"Table: {table_name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nColumn Details:")
    print(df.dtypes)

print("\nTotal Tables:", len(target_tables))

print(
    "Total Records:",
    sum(len(df) for df in target_tables.values())
)


Table: DimRegion
Rows: 21
Columns: 5

Column Details:
region_code            string[python]
region_name                    object
region_abbreviation            object
nuts_1_code                    object
nuts_2_code                    object
dtype: object

Table: DimDate
Rows: 1781
Columns: 8

Column Details:
date_key                int64
full_date      datetime64[ns]
year                    int32
quarter                 int32
month                   int32
month_name             object
day                     int32
day_of_week             int32
dtype: object

Table: DimAgeGroup
Rows: 12
Columns: 2

Column Details:
age_group_key       int64
age_group_label    object
dtype: object

Table: DimNote
Rows: 1617
Columns: 3

Column Details:
note_key      int64
date_key      int64
note_text    object
dtype: object

Table: FactPopulation
Rows: 210
Columns: 6

Column Details:
region_code          string[python]
age_group_key                 int64
male_population               int64
female_popu


## 8. Source-to-Target Mapping

### 8.1 DimRegion – Source-to-Target Mapping

The DimRegion table stores the geographical information of the 21 Italian regions.

The source dataset is the processed Population dataset.

| Source Column | Target Column | Transformation |
|---|---|---|
| codice_regione | region_code | Preserve as string to retain leading zeros |
| denominazione_regione | region_name | Rename only |
| denominazione_regione (if applicable) | region_abbreviation | Verify actual source column before mapping |
| codice_nuts1 | nuts_1_code | Rename only |
| codice_nuts2 | nuts_2_code | Rename only |

**Target Grain:** One row per region.

**Primary Key:** region_code

**Expected Records:** 21


In [44]:

# Inspect source and target columns for DimRegion mapping

print("Source Dataset Columns:")
print(population_codes_check.columns.tolist())

print("\nTarget DimRegion Columns:")
print(dim_region.columns.tolist())

print("\nSource Sample:")
display(population_codes_check.head(3))

print("\nTarget Sample:")
display(dim_region.head(3))


Source Dataset Columns:
['codice_regione', 'codice_nuts_1', 'descrizione_nuts_1', 'codice_nuts_2', 'denominazione_regione', 'sigla_regione', 'latitudine_regione', 'longitudine_regione', 'range_eta', 'totale_genere_maschile', 'totale_genere_femminile', 'totale_generale']

Target DimRegion Columns:
['region_code', 'region_name', 'region_abbreviation', 'nuts_1_code', 'nuts_2_code']

Source Sample:


,codice_regione,codice_nuts_1,descrizione_nuts_1,codice_nuts_2,denominazione_regione,sigla_regione,latitudine_regione,longitudine_regione,range_eta,totale_genere_maschile,totale_genere_femminile,totale_generale
0,13,ITF,Sud,ITF1,Abruzzo,ABR,42.351222,13.398438,0-15,87423,82474,169897
1,13,ITF,Sud,ITF1,Abruzzo,ABR,42.351222,13.398438,16-19,24402,22349,46751
2,13,ITF,Sud,ITF1,Abruzzo,ABR,42.351222,13.398438,20-29,67472,62611,130083



Target Sample:


,region_code,region_name,region_abbreviation,nuts_1_code,nuts_2_code
0,01,Piemonte,PIE,ITC,ITC1
1,02,Valle d'Aosta,VDA,ITC,ITC2
2,03,Lombardia,LOM,ITC,ITC4


In [45]:

# Validate DimRegion source-to-target mapping

dim_region_mapping = {
    "codice_regione": "region_code",
    "denominazione_regione": "region_name",
    "sigla_regione": "region_abbreviation",
    "codice_nuts_1": "nuts_1_code",
    "codice_nuts_2": "nuts_2_code"
}

for source_col, target_col in dim_region_mapping.items():
    print(
        f"{source_col} -> {target_col}:",
        source_col in population_codes_check.columns
        and target_col in dim_region.columns
    )


codice_regione -> region_code: True
denominazione_regione -> region_name: True
sigla_regione -> region_abbreviation: True
codice_nuts_1 -> nuts_1_code: True
codice_nuts_2 -> nuts_2_code: True


In [46]:

# Validate DimDate source-to-target mapping

dim_date_mapping = {
    "data (National)": "full_date",
    "data (Regional)": "full_date",
    "full_date": "date_key",
    "full_date": "year",
    "full_date ": "quarter",
    "full_date  ": "month",
    "full_date   ": "month_name",
    "full_date    ": "day",
    "full_date     ": "day_of_week"
}

print("Target Columns Validation:")

for target_col in dim_date.columns:
    print(f"{target_col}: {target_col in dim_date.columns}")

print("\nDate Dimension Summary:")
print("Rows:", len(dim_date))
print("Unique date keys:", dim_date["date_key"].nunique())
print("Duplicate date keys:", dim_date["date_key"].duplicated().sum())
print("Missing date keys:", dim_date["date_key"].isna().sum())


Target Columns Validation:
date_key: True
full_date: True
year: True
quarter: True
month: True
month_name: True
day: True
day_of_week: True

Date Dimension Summary:
Rows: 1781
Unique date keys: 1781
Duplicate date keys: 0
Missing date keys: 0


In [47]:

# Validate DimAgeGroup source-to-target mapping

print("Source Column Exists:",
      "range_eta" in population_codes_check.columns)

print("Target Columns:")
for col in dim_age_group.columns:
    print(f"{col}: {col in dim_age_group.columns}")

print("\nDimAgeGroup Summary:")
print("Rows:", len(dim_age_group))
print("Unique age labels:", dim_age_group["age_group_label"].nunique())
print("Duplicate age labels:",
      dim_age_group["age_group_label"].duplicated().sum())
print("Duplicate age keys:",
      dim_age_group["age_group_key"].duplicated().sum())
print("Missing age labels:",
      dim_age_group["age_group_label"].isna().sum())
print("Missing age keys:",
      dim_age_group["age_group_key"].isna().sum())


Source Column Exists: True
Target Columns:
age_group_key: True
age_group_label: True

DimAgeGroup Summary:
Rows: 12
Unique age labels: 12
Duplicate age labels: 0
Duplicate age keys: 0
Missing age labels: 0
Missing age keys: 0


In [48]:

# Validate FactPopulation source-to-target mapping

source_columns = [
    "codice_regione",
    "range_eta",
    "totale_genere_maschile",
    "totale_genere_femminile",
    "totale_generale"
]

target_columns = [
    "region_code",
    "age_group_key",
    "male_population",
    "female_population",
    "total_population"
]

print("Source Columns Validation:")
for col in source_columns:
    print(f"{col}: {col in population_codes_check.columns}")

print("\nTarget Columns Validation:")
for col in target_columns:
    print(f"{col}: {col in fact_population.columns}")

print("\nFactPopulation Summary:")
print("Rows:", len(fact_population))
print("Duplicate region-age combinations:",
      fact_population.duplicated(
          subset=["region_code", "age_group_key"]
      ).sum())

print("Missing Values:")
print(fact_population[target_columns].isna().sum())

print("\nPopulation Consistency:")
print(
    "Mismatches:",
    (
        fact_population["male_population"]
        + fact_population["female_population"]
        != fact_population["total_population"]
    ).sum()
)


Source Columns Validation:
codice_regione: True
range_eta: True
totale_genere_maschile: True
totale_genere_femminile: True
totale_generale: True

Target Columns Validation:
region_code: True
age_group_key: True
male_population: True
female_population: True
total_population: True

FactPopulation Summary:
Rows: 210
Duplicate region-age combinations: 0
Missing Values:
region_code          0
age_group_key        0
male_population      0
female_population    0
total_population     0
dtype: int64

Population Consistency:
Mismatches: 0


In [49]:

# Inspect National source and target columns

print("National Source Columns:")
print(national_df.columns.tolist())

print("\nFactNational Target Columns:")
print(fact_national.columns.tolist())

print("\nNational Source Sample:")
display(national_df.head(3))

print("\nFactNational Sample:")
display(fact_national.head(3))


National Source Columns:
['data', 'stato', 'ricoverati_con_sintomi', 'terapia_intensiva', 'totale_ospedalizzati', 'isolamento_domiciliare', 'totale_positivi', 'variazione_totale_positivi', 'nuovi_positivi', 'dimessi_guariti', 'deceduti', 'casi_da_sospetto_diagnostico', 'casi_da_screening', 'totale_casi', 'tamponi', 'casi_testati', 'note', 'ingressi_terapia_intensiva', 'note_test', 'note_casi', 'totale_positivi_test_molecolare', 'totale_positivi_test_antigenico_rapido', 'tamponi_test_molecolare', 'tamponi_test_antigenico_rapido']

FactNational Target Columns:
['date_key', 'symptomatic_hospitalized', 'intensive_care', 'total_hospitalized', 'home_isolation', 'total_positive', 'positive_variation', 'new_positive_cases', 'recovered', 'deaths', 'suspected_diagnosis_cases', 'screening_cases', 'total_cases', 'total_tests', 'tested_cases', 'intensive_care_admissions', 'molecular_positive_cases', 'rapid_antigen_positive_cases', 'molecular_tests', 'rapid_antigen_tests']

National Source Sample:


,data,stato,ricoverati_con_sintomi,terapia_intensiva,totale_ospedalizzati,isolamento_domiciliare,totale_positivi,variazione_totale_positivi,nuovi_positivi,dimessi_guariti,...,tamponi,casi_testati,note,ingressi_terapia_intensiva,note_test,note_casi,totale_positivi_test_molecolare,totale_positivi_test_antigenico_rapido,tamponi_test_molecolare,tamponi_test_antigenico_rapido
0,2020-02-24 18:00:00,ITA,101,26,127,94,221,0,221,1,...,4324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-02-25 18:00:00,ITA,114,35,150,162,311,90,93,1,...,8623,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-02-26 18:00:00,ITA,128,36,164,221,385,74,78,3,...,9587,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



FactNational Sample:


,date_key,symptomatic_hospitalized,intensive_care,total_hospitalized,home_isolation,total_positive,positive_variation,new_positive_cases,recovered,deaths,suspected_diagnosis_cases,screening_cases,total_cases,total_tests,tested_cases,intensive_care_admissions,molecular_positive_cases,rapid_antigen_positive_cases,molecular_tests,rapid_antigen_tests
0,20200224,101,26,127,94,221,0,221,1,7,NaN,NaN,229,4324,NaN,NaN,NaN,NaN,NaN,NaN
1,20200225,114,35,150,162,311,90,93,1,10,NaN,NaN,322,8623,NaN,NaN,NaN,NaN,NaN,NaN
2,20200226,128,36,164,221,385,74,78,3,12,NaN,NaN,400,9587,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:

# Validate FactNational source-to-target mapping

fact_national_mapping = {
    "data": "date_key",
    "ricoverati_con_sintomi": "symptomatic_hospitalized",
    "terapia_intensiva": "intensive_care",
    "totale_ospedalizzati": "total_hospitalized",
    "isolamento_domiciliare": "home_isolation",
    "totale_positivi": "total_positive",
    "variazione_totale_positivi": "positive_variation",
    "nuovi_positivi": "new_positive_cases",
    "dimessi_guariti": "recovered",
    "deceduti": "deaths",
    "casi_da_sospetto_diagnostico": "suspected_diagnosis_cases",
    "casi_da_screening": "screening_cases",
    "totale_casi": "total_cases",
    "tamponi": "total_tests",
    "casi_testati": "tested_cases",
    "ingressi_terapia_intensiva": "intensive_care_admissions",
    "totale_positivi_test_molecolare": "molecular_positive_cases",
    "totale_positivi_test_antigenico_rapido": "rapid_antigen_positive_cases",
    "tamponi_test_molecolare": "molecular_tests",
    "tamponi_test_antigenico_rapido": "rapid_antigen_tests"
}

print("FactNational Mapping Validation:")

for source_col, target_col in fact_national_mapping.items():
    source_exists = source_col in national_df.columns
    target_exists = target_col in fact_national.columns

    print(
        f"{source_col} -> {target_col}: "
        f"{source_exists and target_exists}"
    )

print("\nMapping Summary:")
print("Total mappings:", len(fact_national_mapping))
print(
    "Passed mappings:",
    sum(
        source_col in national_df.columns
        and target_col in fact_national.columns
        for source_col, target_col in fact_national_mapping.items()
    )
)


FactNational Mapping Validation:
data -> date_key: True
ricoverati_con_sintomi -> symptomatic_hospitalized: True
terapia_intensiva -> intensive_care: True
totale_ospedalizzati -> total_hospitalized: True
isolamento_domiciliare -> home_isolation: True
totale_positivi -> total_positive: True
variazione_totale_positivi -> positive_variation: True
nuovi_positivi -> new_positive_cases: True
dimessi_guariti -> recovered: True
deceduti -> deaths: True
casi_da_sospetto_diagnostico -> suspected_diagnosis_cases: True
casi_da_screening -> screening_cases: True
totale_casi -> total_cases: True
tamponi -> total_tests: True
casi_testati -> tested_cases: True
ingressi_terapia_intensiva -> intensive_care_admissions: True
totale_positivi_test_molecolare -> molecular_positive_cases: True
totale_positivi_test_antigenico_rapido -> rapid_antigen_positive_cases: True
tamponi_test_molecolare -> molecular_tests: True
tamponi_test_antigenico_rapido -> rapid_antigen_tests: True

Mapping Summary:
Total mappings:

In [51]:

# Inspect Regional source and target columns

print("Regional Source Columns:")
print(regional_codes_check.columns.tolist())

print("\nFactRegional Target Columns:")
print(fact_regional.columns.tolist())

print("\nRegional Source Sample:")
display(regional_codes_check.head(3))

print("\nFactRegional Sample:")
display(fact_regional.head(3))


Regional Source Columns:
['data', 'stato', 'codice_regione', 'denominazione_regione', 'lat', 'long', 'ricoverati_con_sintomi', 'terapia_intensiva', 'totale_ospedalizzati', 'isolamento_domiciliare', 'totale_positivi', 'variazione_totale_positivi', 'nuovi_positivi', 'dimessi_guariti', 'deceduti', 'casi_da_sospetto_diagnostico', 'casi_da_screening', 'totale_casi', 'tamponi', 'casi_testati', 'note', 'ingressi_terapia_intensiva', 'note_test', 'note_casi', 'totale_positivi_test_molecolare', 'totale_positivi_test_antigenico_rapido', 'tamponi_test_molecolare', 'tamponi_test_antigenico_rapido', 'codice_nuts_1', 'codice_nuts_2']

FactRegional Target Columns:
['region_code', 'date_key', 'symptomatic_hospitalized', 'intensive_care', 'total_hospitalized', 'home_isolation', 'total_positive', 'positive_variation', 'new_positive_cases', 'recovered', 'deaths', 'suspected_diagnosis_cases', 'screening_cases', 'total_cases', 'total_tests', 'tested_cases', 'intensive_care_admissions', 'molecular_positive_c

,data,stato,codice_regione,denominazione_regione,lat,long,ricoverati_con_sintomi,terapia_intensiva,totale_ospedalizzati,isolamento_domiciliare,...,note,ingressi_terapia_intensiva,note_test,note_casi,totale_positivi_test_molecolare,totale_positivi_test_antigenico_rapido,tamponi_test_molecolare,tamponi_test_antigenico_rapido,codice_nuts_1,codice_nuts_2
0,2020-02-24 18:00:00,ITA,13,Abruzzo,42.351222,13.398438,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-02-24 18:00:00,ITA,17,Basilicata,40.639471,15.805148,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-02-24 18:00:00,ITA,18,Calabria,38.905976,16.594402,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



FactRegional Sample:


,region_code,date_key,symptomatic_hospitalized,intensive_care,total_hospitalized,home_isolation,total_positive,positive_variation,new_positive_cases,recovered,...,suspected_diagnosis_cases,screening_cases,total_cases,total_tests,tested_cases,intensive_care_admissions,molecular_positive_cases,rapid_antigen_positive_cases,molecular_tests,rapid_antigen_tests
0,13,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,5,NaN,NaN,NaN,NaN,NaN,NaN
1,17,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,18,20200224,0,0,0,0,0,0,0,0,...,NaN,NaN,0,1,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:

# Validate FactRegional source-to-target mapping

fact_regional_mapping = {
    "codice_regione": "region_code",
    "data": "date_key",
    "ricoverati_con_sintomi": "symptomatic_hospitalized",
    "terapia_intensiva": "intensive_care",
    "totale_ospedalizzati": "total_hospitalized",
    "isolamento_domiciliare": "home_isolation",
    "totale_positivi": "total_positive",
    "variazione_totale_positivi": "positive_variation",
    "nuovi_positivi": "new_positive_cases",
    "dimessi_guariti": "recovered",
    "deceduti": "deaths",
    "casi_da_sospetto_diagnostico": "suspected_diagnosis_cases",
    "casi_da_screening": "screening_cases",
    "totale_casi": "total_cases",
    "tamponi": "total_tests",
    "casi_testati": "tested_cases",
    "ingressi_terapia_intensiva": "intensive_care_admissions",
    "totale_positivi_test_molecolare": "molecular_positive_cases",
    "totale_positivi_test_antigenico_rapido": "rapid_antigen_positive_cases",
    "tamponi_test_molecolare": "molecular_tests",
    "tamponi_test_antigenico_rapido": "rapid_antigen_tests"
}

print("FactRegional Mapping Validation:")

for source_col, target_col in fact_regional_mapping.items():
    source_exists = source_col in regional_codes_check.columns
    target_exists = target_col in fact_regional.columns

    print(
        f"{source_col} -> {target_col}: "
        f"{source_exists and target_exists}"
    )

print("\nMapping Summary:")
print("Total mappings:", len(fact_regional_mapping))

passed = sum(
    source_col in regional_codes_check.columns
    and target_col in fact_regional.columns
    for source_col, target_col in fact_regional_mapping.items()
)

print("Passed mappings:", passed)


FactRegional Mapping Validation:
codice_regione -> region_code: True
data -> date_key: True
ricoverati_con_sintomi -> symptomatic_hospitalized: True
terapia_intensiva -> intensive_care: True
totale_ospedalizzati -> total_hospitalized: True
isolamento_domiciliare -> home_isolation: True
totale_positivi -> total_positive: True
variazione_totale_positivi -> positive_variation: True
nuovi_positivi -> new_positive_cases: True
dimessi_guariti -> recovered: True
deceduti -> deaths: True
casi_da_sospetto_diagnostico -> suspected_diagnosis_cases: True
casi_da_screening -> screening_cases: True
totale_casi -> total_cases: True
tamponi -> total_tests: True
casi_testati -> tested_cases: True
ingressi_terapia_intensiva -> intensive_care_admissions: True
totale_positivi_test_molecolare -> molecular_positive_cases: True
totale_positivi_test_antigenico_rapido -> rapid_antigen_positive_cases: True
tamponi_test_molecolare -> molecular_tests: True
tamponi_test_antigenico_rapido -> rapid_antigen_tests: Tr

In [53]:

# Inspect Notes source and target columns

print("Notes Source Columns:")
print(notes_df.columns.tolist())

print("\nDimNote Target Columns:")
print(dim_note.columns.tolist())

print("\nNotes Source Sample:")
display(notes_df.head(5))

print("\nDimNote Sample:")
display(dim_note.head(5))


Notes Source Columns:
['data', 'note']

DimNote Target Columns:
['note_key', 'date_key', 'note_text']

Notes Source Sample:


,data,note
0,2020-03-07 17:00:00,La Provincia di Brescia ha comunicato dati par...
1,2020-03-10 17:00:00,La Regione Lombardia ha comunicato dati parziali
2,2020-06-01 17:00:00,[altre date in fase di caricamento]
3,2020-06-25 17:00:00,La Regione Marche ha effettuato un ricalcolo s...
4,2020-06-26 17:00:00,La Regione Emilia Romagna ha effettuato un ric...



DimNote Sample:


,note_key,date_key,note_text
0,1,20200307,La Provincia di Brescia ha comunicato dati par...
1,2,20200310,La Regione Lombardia ha comunicato dati parziali
2,3,20200601,[altre date in fase di caricamento]
3,4,20200625,La Regione Marche ha effettuato un ricalcolo s...
4,5,20200626,La Regione Emilia Romagna ha effettuato un ric...


In [54]:

# Validate DimNote source-to-target mapping

dim_note_mapping = {
    "data": "date_key",
    "note": "note_text"
}

print("DimNote Mapping Validation:")

for source_col, target_col in dim_note_mapping.items():
    source_exists = source_col in notes_df.columns
    target_exists = target_col in dim_note.columns

    print(
        f"{source_col} -> {target_col}: "
        f"{source_exists and target_exists}"
    )

print("\nGenerated Key Validation:")
print("note_key exists:", "note_key" in dim_note.columns)
print("Unique note keys:", dim_note["note_key"].nunique())
print("Duplicate note keys:", dim_note["note_key"].duplicated().sum())
print("Missing note keys:", dim_note["note_key"].isna().sum())

print("\nDimNote Summary:")
print("Rows:", len(dim_note))
print("Unique dates:", dim_note["date_key"].nunique())
print("Missing date keys:", dim_note["date_key"].isna().sum())
print("Missing note texts:", dim_note["note_text"].isna().sum())


DimNote Mapping Validation:
data -> date_key: True
note -> note_text: True

Generated Key Validation:
note_key exists: True
Unique note keys: 1617
Duplicate note keys: 0
Missing note keys: 0

DimNote Summary:
Rows: 1617
Unique dates: 1617
Missing date keys: 0
Missing note texts: 14


In [55]:

# Final Source-to-Target Mapping Audit

target_tables = {
    "DimRegion": {
        "dataframe": dim_region,
        "expected_rows": 21,
        "expected_columns": 5
    },
    "DimDate": {
        "dataframe": dim_date,
        "expected_rows": 1781,
        "expected_columns": 8
    },
    "DimAgeGroup": {
        "dataframe": dim_age_group,
        "expected_rows": 12,
        "expected_columns": 2
    },
    "DimNote": {
        "dataframe": dim_note,
        "expected_rows": 1617,
        "expected_columns": 3
    },
    "FactPopulation": {
        "dataframe": fact_population[
            [
                "region_code",
                "age_group_key",
                "male_population",
                "female_population",
                "total_population"
            ]
        ],
        "expected_rows": 210,
        "expected_columns": 5
    },
    "FactNational": {
        "dataframe": fact_national,
        "expected_rows": 1781,
        "expected_columns": 20
    },
    "FactRegional": {
        "dataframe": fact_regional,
        "expected_rows": 37401,
        "expected_columns": 21
    }
}

audit_results = []

for table_name, details in target_tables.items():
    df = details["dataframe"]

    audit_results.append({
        "Table": table_name,
        "Actual Rows": len(df),
        "Expected Rows": details["expected_rows"],
        "Actual Columns": len(df.columns),
        "Expected Columns": details["expected_columns"],
        "Rows Check": len(df) == details["expected_rows"],
        "Columns Check": len(df.columns) == details["expected_columns"]
    })

mapping_audit_df = pd.DataFrame(audit_results)

display(mapping_audit_df)

print("\nFinal Audit Summary:")
print("Total Tables:", len(mapping_audit_df))
print("Total Records:", mapping_audit_df["Actual Rows"].sum())
print("Rows Checks Passed:", mapping_audit_df["Rows Check"].sum())
print("Columns Checks Passed:", mapping_audit_df["Columns Check"].sum())
print(
    "Overall Audit:",
    "PASSED" if (
        mapping_audit_df["Rows Check"].all()
        and mapping_audit_df["Columns Check"].all()
    ) else "FAILED"
)


,Table,Actual Rows,Expected Rows,Actual Columns,Expected Columns,Rows Check,Columns Check
0,DimRegion,21,21,5,5,True,True
1,DimDate,1781,1781,8,8,True,True
2,DimAgeGroup,12,12,2,2,True,True
3,DimNote,1617,1617,3,3,True,True
4,FactPopulation,210,210,5,5,True,True
5,FactNational,1781,1781,20,20,True,True
6,FactRegional,37401,37401,21,21,True,True



Final Audit Summary:
Total Tables: 7
Total Records: 42823
Rows Checks Passed: 7
Columns Checks Passed: 7
Overall Audit: PASSED



# 9. SQL Server Data Type Mapping

This section defines the SQL Server data types for all columns in the seven target tables.

The mapping considers:
- Current Pandas data types.
- Actual data values and value ranges.
- NULL handling.
- String length and Unicode requirements.
- Primary key and foreign key compatibility.
- Numeric precision and storage requirements.

The objective is to establish a consistent SQL Server schema before table creation and data loading.

### 9.1 Data Type Mapping – Design Principles

1. Use appropriate integer types for whole-number measures.
2. Preserve missing values as SQL NULL.
3. Use DATE for calendar dates where time information is not required.
4. Use NVARCHAR for text requiring Unicode support.
5. Ensure foreign key and referenced primary key data types are compatible.
6. Select numeric types based on actual value ranges rather than Pandas types alone.


In [56]:

# Inspect numeric column ranges across target tables

numeric_tables = {
    "DimDate": dim_date,
    "DimAgeGroup": dim_age_group,
    "FactPopulation": fact_population,
    "FactNational": fact_national,
    "FactRegional": fact_regional
}

for table_name, df in numeric_tables.items():

    print(f"\n{'='*60}")
    print(f"Table: {table_name}")
    print(f"{'='*60}")

    numeric_columns = df.select_dtypes(
        include=["number"]
    ).columns

    for col in numeric_columns:
        print(f"\nColumn: {col}")
        print("Min:", df[col].min())
        print("Max:", df[col].max())
        print("Missing:", df[col].isna().sum())



Table: DimDate

Column: date_key
Min: 20200224
Max: 20250108
Missing: 0

Column: year
Min: 2020
Max: 2025
Missing: 0

Column: quarter
Min: 1
Max: 4
Missing: 0

Column: month
Min: 1
Max: 12
Missing: 0

Column: day
Min: 1
Max: 31
Missing: 0

Column: day_of_week
Min: 1
Max: 7
Missing: 0

Table: DimAgeGroup

Column: age_group_key
Min: 1
Max: 12
Missing: 0

Table: FactPopulation

Column: age_group_key
Min: 1
Max: 12
Missing: 0

Column: male_population
Min: 457
Max: 805221
Missing: 0

Column: female_population
Min: 1307
Max: 802504
Missing: 0

Column: total_population
Min: 1764
Max: 1607725
Missing: 0

Column: calculated_total
Min: 1764
Max: 1607725
Missing: 0

Table: FactNational

Column: date_key
Min: 20200224
Max: 20250108
Missing: 0

Column: symptomatic_hospitalized
Min: 101
Max: 34697
Missing: 0

Column: intensive_care
Min: 17
Max: 4068
Missing: 0

Column: total_hospitalized
Min: 127
Max: 38507
Missing: 0

Column: home_isolation
Min: 94
Max: 2713594
Missing: 0

Column: total_positive
M

In [57]:

# Validate whether float64 columns contain fractional values

tables_to_check = {
    "FactNational": fact_national,
    "FactRegional": fact_regional
}

for table_name, df in tables_to_check.items():

    print(f"\n{'='*60}")
    print(f"Table: {table_name}")
    print(f"{'='*60}")

    float_columns = df.select_dtypes(
        include=["float"]
    ).columns

    for col in float_columns:

        non_null_values = df[col].dropna()

        fractional_count = (
            non_null_values % 1 != 0
        ).sum()

        print(f"\nColumn: {col}")
        print("Non-null values:", len(non_null_values))
        print("Fractional values:", fractional_count)
        print("Contains only whole numbers:",
              fractional_count == 0)



Table: FactNational

Column: suspected_diagnosis_cases
Non-null values: 162
Fractional values: 0
Contains only whole numbers: True

Column: screening_cases
Non-null values: 162
Fractional values: 0
Contains only whole numbers: True

Column: tested_cases
Non-null values: 1726
Fractional values: 0
Contains only whole numbers: True

Column: intensive_care_admissions
Non-null values: 1498
Fractional values: 0
Contains only whole numbers: True

Column: molecular_positive_cases
Non-null values: 1455
Fractional values: 0
Contains only whole numbers: True

Column: rapid_antigen_positive_cases
Non-null values: 1455
Fractional values: 0
Contains only whole numbers: True

Column: molecular_tests
Non-null values: 1455
Fractional values: 0
Contains only whole numbers: True

Column: rapid_antigen_tests
Non-null values: 1455
Fractional values: 0
Contains only whole numbers: True

Table: FactRegional

Column: suspected_diagnosis_cases
Non-null values: 3402
Fractional values: 0
Contains only whole num

In [58]:

# Inspect string lengths in DimRegion

text_columns = [
    "region_code",
    "region_name",
    "region_abbreviation",
    "nuts_1_code",
    "nuts_2_code"
]

for col in text_columns:
    lengths = dim_region[col].dropna().astype(str).str.len()

    print(f"\nColumn: {col}")
    print("Maximum Length:", lengths.max())
    print("Missing Values:", dim_region[col].isna().sum())
    print("Unique Values:", dim_region[col].nunique())



Column: region_code
Maximum Length: 2
Missing Values: 0
Unique Values: 21

Column: region_name
Maximum Length: 21
Missing Values: 0
Unique Values: 21

Column: region_abbreviation
Maximum Length: 3
Missing Values: 0
Unique Values: 21

Column: nuts_1_code
Maximum Length: 3
Missing Values: 0
Unique Values: 5

Column: nuts_2_code
Maximum Length: 4
Missing Values: 0
Unique Values: 21


In [59]:

# Inspect DimDate values and text lengths

print("DimDate Text Column:")
print("Maximum month_name length:",
      dim_date["month_name"].dropna().astype(str).str.len().max())

print("Missing month_name:",
      dim_date["month_name"].isna().sum())

print("\nDate Key:")
print("Minimum:", dim_date["date_key"].min())
print("Maximum:", dim_date["date_key"].max())
print("Unique:", dim_date["date_key"].nunique())

print("\nDate Range:")
print("Minimum year:", dim_date["year"].min())
print("Maximum year:", dim_date["year"].max())

print("\nCalendar Attributes:")
for col in ["quarter", "month", "day", "day_of_week"]:
    print(
        f"{col}: Min={dim_date[col].min()}, "
        f"Max={dim_date[col].max()}, "
        f"Missing={dim_date[col].isna().sum()}"
    )

print("\nFull Date:")
print("Missing:", dim_date["full_date"].isna().sum())
print("Duplicates:", dim_date["full_date"].duplicated().sum())


DimDate Text Column:
Maximum month_name length: 9
Missing month_name: 0

Date Key:
Minimum: 20200224
Maximum: 20250108
Unique: 1781

Date Range:
Minimum year: 2020
Maximum year: 2025

Calendar Attributes:
quarter: Min=1, Max=4, Missing=0
month: Min=1, Max=12, Missing=0
day: Min=1, Max=31, Missing=0
day_of_week: Min=1, Max=7, Missing=0

Full Date:
Missing: 0
Duplicates: 0


In [60]:

# Inspect DimAgeGroup values and string lengths

print("Age Group Key:")
print("Minimum:", dim_age_group["age_group_key"].min())
print("Maximum:", dim_age_group["age_group_key"].max())
print("Unique Keys:", dim_age_group["age_group_key"].nunique())
print("Missing Keys:", dim_age_group["age_group_key"].isna().sum())

print("\nAge Group Label:")
print("Maximum Length:",
      dim_age_group["age_group_label"].dropna().astype(str).str.len().max())
print("Unique Labels:", dim_age_group["age_group_label"].nunique())
print("Missing Labels:", dim_age_group["age_group_label"].isna().sum())

print("\nAll Age Group Labels:")
print(sorted(dim_age_group["age_group_label"].dropna().unique()))


Age Group Key:
Minimum: 1
Maximum: 12
Unique Keys: 12
Missing Keys: 0

Age Group Label:
Maximum Length: 5
Unique Labels: 12
Missing Labels: 0

All Age Group Labels:
['0-15', '05-11', '12-19', '16-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']


In [61]:

# Inspect DimNote values and text lengths

note_lengths = (
    dim_note["note_text"]
    .dropna()
    .astype(str)
    .str.len()
)

print("Note Key:")
print("Minimum:", dim_note["note_key"].min())
print("Maximum:", dim_note["note_key"].max())
print("Unique:", dim_note["note_key"].nunique())
print("Missing:", dim_note["note_key"].isna().sum())

print("\nDate Key:")
print("Missing:", dim_note["date_key"].isna().sum())
print("Unique Dates:", dim_note["date_key"].nunique())

print("\nNote Text:")
print("Maximum Length:", note_lengths.max())
print("Minimum Length:", note_lengths.min())
print("Missing Values:", dim_note["note_text"].isna().sum())
print("Non-null Notes:", dim_note["note_text"].notna().sum())


Note Key:
Minimum: 1
Maximum: 1617
Unique: 1617
Missing: 0

Date Key:
Missing: 0
Unique Dates: 1617

Note Text:
Maximum Length: 2325
Minimum Length: 35
Missing Values: 14
Non-null Notes: 1603


In [62]:

# Inspect FactPopulation SQL Server data type requirements

population_columns = [
    "male_population",
    "female_population",
    "total_population"
]

print("Region Code:")
print("Maximum Length:",
      fact_population["region_code"].astype(str).str.len().max())
print("Missing:",
      fact_population["region_code"].isna().sum())

print("\nAge Group Key:")
print("Minimum:", fact_population["age_group_key"].min())
print("Maximum:", fact_population["age_group_key"].max())
print("Missing:", fact_population["age_group_key"].isna().sum())

print("\nPopulation Measures:")

for col in population_columns:
    print(f"\nColumn: {col}")
    print("Minimum:", fact_population[col].min())
    print("Maximum:", fact_population[col].max())
    print("Missing:", fact_population[col].isna().sum())
    print("Fractional Values:",
          (fact_population[col] % 1 != 0).sum())

print("\nComposite Key:")
print(
    "Duplicate combinations:",
    fact_population.duplicated(
        subset=["region_code", "age_group_key"]
    ).sum()
)


Region Code:
Maximum Length: 2
Missing: 0

Age Group Key:
Minimum: 1
Maximum: 12
Missing: 0

Population Measures:

Column: male_population
Minimum: 457
Maximum: 805221
Missing: 0
Fractional Values: 0

Column: female_population
Minimum: 1307
Maximum: 802504
Missing: 0
Fractional Values: 0

Column: total_population
Minimum: 1764
Maximum: 1607725
Missing: 0
Fractional Values: 0

Composite Key:
Duplicate combinations: 0


In [63]:

# Inspect FactNational SQL Server data type requirements

print("FactNational Date Key:")
print("Minimum:", fact_national["date_key"].min())
print("Maximum:", fact_national["date_key"].max())
print("Missing:", fact_national["date_key"].isna().sum())
print("Duplicates:", fact_national["date_key"].duplicated().sum())

print("\nFactNational Measures:")

for col in fact_national.columns:
    if col == "date_key":
        continue

    print(f"\nColumn: {col}")
    print("Minimum:", fact_national[col].min())
    print("Maximum:", fact_national[col].max())
    print("Missing:", fact_national[col].isna().sum())

    non_null = fact_national[col].dropna()
    print("Fractional Values:", (non_null % 1 != 0).sum())


FactNational Date Key:
Minimum: 20200224
Maximum: 20250108
Missing: 0
Duplicates: 0

FactNational Measures:

Column: symptomatic_hospitalized
Minimum: 101
Maximum: 34697
Missing: 0
Fractional Values: 0

Column: intensive_care
Minimum: 17
Maximum: 4068
Missing: 0
Fractional Values: 0

Column: total_hospitalized
Minimum: 127
Maximum: 38507
Missing: 0
Fractional Values: 0

Column: home_isolation
Minimum: 94
Maximum: 2713594
Missing: 0
Fractional Values: 0

Column: total_positive
Minimum: 221
Maximum: 2734906
Missing: 0
Fractional Values: 0

Column: positive_variation
Minimum: -116092
Maximum: 172462
Missing: 0
Fractional Values: 0

Column: new_positive_cases
Minimum: 30
Maximum: 220532
Missing: 0
Fractional Values: 0

Column: recovered
Minimum: 1
Maximum: 26562666
Missing: 0
Fractional Values: 0

Column: deaths
Minimum: 7
Maximum: 198683
Missing: 0
Fractional Values: 0

Column: suspected_diagnosis_cases
Minimum: 0.0
Maximum: 988470.0
Missing: 1619
Fractional Values: 0

Column: screening_c

In [64]:

# Inspect FactRegional SQL Server data type requirements

print("Region Code:")
print("Maximum Length:",
      fact_regional["region_code"].astype(str).str.len().max())
print("Missing:", fact_regional["region_code"].isna().sum())

print("\nDate Key:")
print("Minimum:", fact_regional["date_key"].min())
print("Maximum:", fact_regional["date_key"].max())
print("Missing:", fact_regional["date_key"].isna().sum())

print("\nComposite Key:")
print(
    "Duplicate combinations:",
    fact_regional.duplicated(
        subset=["region_code", "date_key"]
    ).sum()
)

print("\nRegional Measures:")

for col in fact_regional.columns:
    if col in ["region_code", "date_key"]:
        continue

    non_null = fact_regional[col].dropna()

    print(f"\nColumn: {col}")
    print("Minimum:", non_null.min())
    print("Maximum:", non_null.max())
    print("Missing:", fact_regional[col].isna().sum())
    print("Fractional Values:", (non_null % 1 != 0).sum())


Region Code:
Maximum Length: 2
Missing: 0

Date Key:
Minimum: 20200224
Maximum: 20250108
Missing: 0

Composite Key:
Duplicate combinations: 0

Regional Measures:

Column: symptomatic_hospitalized
Minimum: 0
Maximum: 12077
Missing: 0
Fractional Values: 0

Column: intensive_care
Minimum: 0
Maximum: 1381
Missing: 0
Fractional Values: 0

Column: total_hospitalized
Minimum: 0
Maximum: 13328
Missing: 0
Fractional Values: 0

Column: home_isolation
Minimum: 0
Maximum: 574548
Missing: 0
Fractional Values: 0

Column: total_positive
Minimum: 0
Maximum: 578257
Missing: 0
Fractional Values: 0

Column: positive_variation
Minimum: -50797
Maximum: 47483
Missing: 0
Fractional Values: 0

Column: new_positive_cases
Minimum: -229
Maximum: 52693
Missing: 0
Fractional Values: 0

Column: recovered
Minimum: 0
Maximum: 4342249
Missing: 0
Fractional Values: 0

Column: deaths
Minimum: 0
Maximum: 48739
Missing: 0
Fractional Values: 0

Column: suspected_diagnosis_cases
Minimum: 0.0
Maximum: 305002.0
Missing: 33999


### 9.10 Final SQL Server Data Type Audit

This section consolidates the SQL Server data type decisions for all seven target tables.

The audit reviews:
- SQL Server data type assignments.
- NULL and NOT NULL constraints.
- Primary key definitions.
- Foreign key compatibility.
- Numeric range suitability.
- String length suitability.
- Preservation of missing and negative values.

No SQL Server tables are created in this section.

The finalized schema will be used as the basis for the subsequent SQL Server DDL implementation.


In [65]:

# Define SQL Server schema mapping for all target tables

sql_schema_mapping = {

    "DimRegion": {
        "region_code": ("VARCHAR(2)", "NOT NULL", "PRIMARY KEY"),
        "region_name": ("NVARCHAR(100)", "NOT NULL", ""),
        "region_abbreviation": ("VARCHAR(10)", "NOT NULL", ""),
        "nuts_1_code": ("VARCHAR(10)", "NOT NULL", ""),
        "nuts_2_code": ("VARCHAR(10)", "NOT NULL", "")
    },

    "DimDate": {
        "date_key": ("INT", "NOT NULL", "PRIMARY KEY"),
        "full_date": ("DATE", "NOT NULL", "UNIQUE"),
        "year": ("SMALLINT", "NOT NULL", ""),
        "quarter": ("TINYINT", "NOT NULL", ""),
        "month": ("TINYINT", "NOT NULL", ""),
        "month_name": ("NVARCHAR(20)", "NOT NULL", ""),
        "day": ("TINYINT", "NOT NULL", ""),
        "day_of_week": ("TINYINT", "NOT NULL", "")
    },

    "DimAgeGroup": {
        "age_group_key": ("INT", "NOT NULL", "PRIMARY KEY"),
        "age_group_label": ("NVARCHAR(20)", "NOT NULL", "UNIQUE")
    },

    "DimNote": {
        "note_key": ("INT", "NOT NULL", "PRIMARY KEY"),
        "date_key": ("INT", "NOT NULL", "FOREIGN KEY"),
        "note_text": ("NVARCHAR(MAX)", "NULL", "")
    },

    "FactPopulation": {
        "region_code": ("VARCHAR(2)", "NOT NULL", "COMPOSITE PRIMARY KEY, FOREIGN KEY"),
        "age_group_key": ("INT", "NOT NULL", "COMPOSITE PRIMARY KEY, FOREIGN KEY"),
        "male_population": ("INT", "NOT NULL", ""),
        "female_population": ("INT", "NOT NULL", ""),
        "total_population": ("INT", "NOT NULL", "")
    },

    "FactNational": {
        "date_key": ("INT", "NOT NULL", "PRIMARY KEY"),
        **{
            col: (
                "INT",
                "NULL" if fact_national[col].isna().any() else "NOT NULL",
                ""
            )
            for col in fact_national.columns
            if col != "date_key"
        }
    },

    "FactRegional": {
        "region_code": ("VARCHAR(2)", "NOT NULL", "COMPOSITE PRIMARY KEY, FOREIGN KEY"),
        "date_key": ("INT", "NOT NULL", "COMPOSITE PRIMARY KEY, FOREIGN KEY"),
        **{
            col: (
                "INT",
                "NULL" if fact_regional[col].isna().any() else "NOT NULL",
                ""
            )
            for col in fact_regional.columns
            if col not in ["region_code", "date_key"]
        }
    }
}

schema_records = []

for table_name, columns in sql_schema_mapping.items():
    for column_name, (sql_type, nullable, constraint) in columns.items():
        schema_records.append({
            "Table": table_name,
            "Column": column_name,
            "SQL Server Type": sql_type,
            "Nullable": nullable,
            "Constraint": constraint
        })

sql_schema_df = pd.DataFrame(schema_records)

display(sql_schema_df)

print("\nSchema Summary:")
print("Total Tables:", sql_schema_df["Table"].nunique())
print("Total Columns:", len(sql_schema_df))
print("NULL Columns:", (sql_schema_df["Nullable"] == "NULL").sum())
print("NOT NULL Columns:", (sql_schema_df["Nullable"] == "NOT NULL").sum())


,Table,Column,SQL Server Type,Nullable,Constraint
0,DimRegion,region_code,VARCHAR(2),NOT NULL,PRIMARY KEY
1,DimRegion,region_name,NVARCHAR(100),NOT NULL,
2,DimRegion,region_abbreviation,VARCHAR(10),NOT NULL,
3,DimRegion,nuts_1_code,VARCHAR(10),NOT NULL,
4,DimRegion,nuts_2_code,VARCHAR(10),NOT NULL,
...,...,...,...,...,...
59,FactRegional,intensive_care_admissions,INT,NULL,
60,FactRegional,molecular_positive_cases,INT,NULL,
61,FactRegional,rapid_antigen_positive_cases,INT,NULL,
62,FactRegional,molecular_tests,INT,NULL,



Schema Summary:
Total Tables: 7
Total Columns: 64
NULL Columns: 17
NOT NULL Columns: 47


In [66]:
foreign_key_mapping = [
    ("FactPopulation", "region_code", "DimRegion", "region_code"),
    ("FactPopulation", "age_group_key", "DimAgeGroup", "age_group_key"),
    ("FactNational", "date_key", "DimDate", "date_key"),
    ("FactRegional", "region_code", "DimRegion", "region_code"),
    ("FactRegional", "date_key", "DimDate", "date_key"),
    ("DimNote", "date_key", "DimDate", "date_key")
]

foreign_key_audit = []

for child_table, child_column, parent_table, parent_column in foreign_key_mapping:

    child_type = sql_schema_mapping[child_table][child_column][0]
    parent_type = sql_schema_mapping[parent_table][parent_column][0]

    foreign_key_audit.append({
        "Child Table": child_table,
        "Child Column": child_column,
        "Parent Table": parent_table,
        "Parent Column": parent_column,
        "Child SQL Type": child_type,
        "Parent SQL Type": parent_type,
        "Type Compatible": child_type == parent_type
    })

foreign_key_audit_df = pd.DataFrame(foreign_key_audit)

display(foreign_key_audit_df)

print(
    "\nCompatible Foreign Keys:",
    foreign_key_audit_df["Type Compatible"].sum(),
    "/",
    len(foreign_key_audit_df)
)

print(
    "Overall FK Type Audit:",
    "PASSED" if foreign_key_audit_df["Type Compatible"].all() else "FAILED"
)

,Child Table,Child Column,Parent Table,Parent Column,Child SQL Type,Parent SQL Type,Type Compatible
0,FactPopulation,region_code,DimRegion,region_code,VARCHAR(2),VARCHAR(2),True
1,FactPopulation,age_group_key,DimAgeGroup,age_group_key,INT,INT,True
2,FactNational,date_key,DimDate,date_key,INT,INT,True
3,FactRegional,region_code,DimRegion,region_code,VARCHAR(2),VARCHAR(2),True
4,FactRegional,date_key,DimDate,date_key,INT,INT,True
5,DimNote,date_key,DimDate,date_key,INT,INT,True



Compatible Foreign Keys: 6 / 6
Overall FK Type Audit: PASSED


In [67]:
primary_key_checks = {
    "DimRegion": {
        "df": dim_region,
        "columns": ["region_code"]
    },
    "DimDate": {
        "df": dim_date,
        "columns": ["date_key"]
    },
    "DimAgeGroup": {
        "df": dim_age_group,
        "columns": ["age_group_key"]
    },
    "DimNote": {
        "df": dim_note,
        "columns": ["note_key"]
    },
    "FactPopulation": {
        "df": fact_population,
        "columns": ["region_code", "age_group_key"]
    },
    "FactNational": {
        "df": fact_national,
        "columns": ["date_key"]
    },
    "FactRegional": {
        "df": fact_regional,
        "columns": ["region_code", "date_key"]
    }
}

primary_key_audit = []

for table_name, config in primary_key_checks.items():

    df = config["df"]
    pk_columns = config["columns"]

    missing_count = df[pk_columns].isna().any(axis=1).sum()
    duplicate_count = df.duplicated(subset=pk_columns).sum()

    primary_key_audit.append({
        "Table": table_name,
        "Primary Key": ", ".join(pk_columns),
        "Missing Key Rows": missing_count,
        "Duplicate Key Rows": duplicate_count,
        "Primary Key Valid": missing_count == 0 and duplicate_count == 0
    })

primary_key_audit_df = pd.DataFrame(primary_key_audit)

display(primary_key_audit_df)

print(
    "\nValid Primary Keys:",
    primary_key_audit_df["Primary Key Valid"].sum(),
    "/",
    len(primary_key_audit_df)
)

print(
    "Overall Primary Key Audit:",
    "PASSED" if primary_key_audit_df["Primary Key Valid"].all() else "FAILED"
)

,Table,Primary Key,Missing Key Rows,Duplicate Key Rows,Primary Key Valid
0,DimRegion,region_code,0,0,True
1,DimDate,date_key,0,0,True
2,DimAgeGroup,age_group_key,0,0,True
3,DimNote,note_key,0,0,True
4,FactPopulation,"region_code, age_group_key",0,0,True
5,FactNational,date_key,0,0,True
6,FactRegional,"region_code, date_key",0,0,True



Valid Primary Keys: 7 / 7
Overall Primary Key Audit: PASSED


In [68]:
unique_constraints = {
    "DimDate": {
        "df": dim_date,
        "column": "full_date"
    },
    "DimAgeGroup": {
        "df": dim_age_group,
        "column": "age_group_label"
    }
}

unique_audit = []

for table_name, config in unique_constraints.items():

    df = config["df"]
    column = config["column"]

    missing_count = df[column].isna().sum()
    duplicate_count = df[column].duplicated().sum()

    unique_audit.append({
        "Table": table_name,
        "Column": column,
        "Missing Values": missing_count,
        "Duplicate Values": duplicate_count,
        "Unique Constraint Valid": (
            missing_count == 0 and duplicate_count == 0
        )
    })

unique_audit_df = pd.DataFrame(unique_audit)

display(unique_audit_df)

print(
    "\nValid Unique Constraints:",
    unique_audit_df["Unique Constraint Valid"].sum(),
    "/",
    len(unique_audit_df)
)

print(
    "Overall Unique Constraint Audit:",
    "PASSED" if unique_audit_df["Unique Constraint Valid"].all() else "FAILED"
)

,Table,Column,Missing Values,Duplicate Values,Unique Constraint Valid
0,DimDate,full_date,0,0,True
1,DimAgeGroup,age_group_label,0,0,True



Valid Unique Constraints: 2 / 2
Overall Unique Constraint Audit: PASSED


## Section 10 – Final Data Transformation and Load Preparation

This section prepares the final transformed datasets for SQL Server loading.

The final datasets will:
- Follow the approved target schema.
- Contain only the required target columns.
- Preserve the approved column names and order.
- Exclude temporary validation columns.
- Preserve missing values without replacing them with zero.
- Retain the validated primary and foreign key values.

The original extracted and transformed DataFrames will remain unchanged.

In [69]:
# Define the final target column order for each table

target_columns = {
    "DimRegion": [
        "region_code",
        "region_name",
        "region_abbreviation",
        "nuts_1_code",
        "nuts_2_code"
    ],

    "DimDate": [
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week"
    ],

    "DimAgeGroup": [
        "age_group_key",
        "age_group_label"
    ],

    "DimNote": [
        "note_key",
        "date_key",
        "note_text"
    ],

    "FactPopulation": [
        "region_code",
        "age_group_key",
        "male_population",
        "female_population",
        "total_population"
    ],

    "FactNational": list(fact_national.columns),

    "FactRegional": list(fact_regional.columns)
}

# Create final DataFrames without modifying the original ones

final_datasets = {
    "DimRegion": dim_region[target_columns["DimRegion"]].copy(),
    "DimDate": dim_date[target_columns["DimDate"]].copy(),
    "DimAgeGroup": dim_age_group[target_columns["DimAgeGroup"]].copy(),
    "DimNote": dim_note[target_columns["DimNote"]].copy(),
    "FactPopulation": fact_population[target_columns["FactPopulation"]].copy(),
    "FactNational": fact_national[target_columns["FactNational"]].copy(),
    "FactRegional": fact_regional[target_columns["FactRegional"]].copy()
}

print("Final datasets created successfully.")

Final datasets created successfully.


In [70]:
expected_shapes = {
    "DimRegion": (21, 5),
    "DimDate": (1781, 8),
    "DimAgeGroup": (12, 2),
    "DimNote": (1617, 3),
    "FactPopulation": (210, 5),
    "FactNational": (1781, 20),
    "FactRegional": (37401, 21)
}

structural_audit = []

for table_name, df in final_datasets.items():

    expected_rows, expected_columns_count = expected_shapes[table_name]

    expected_column_names = target_columns[table_name]

    structural_audit.append({
        "Table": table_name,
        "Actual Rows": len(df),
        "Expected Rows": expected_rows,
        "Actual Columns": len(df.columns),
        "Expected Columns": expected_columns_count,
        "Column Order Valid": list(df.columns) == expected_column_names,
        "Shape Valid": df.shape == expected_shapes[table_name]
    })

structural_audit_df = pd.DataFrame(structural_audit)

display(structural_audit_df)

print("\nValid Structures:",
      structural_audit_df["Shape Valid"].sum(), "/", len(structural_audit_df))

print("Valid Column Orders:",
      structural_audit_df["Column Order Valid"].sum(), "/", len(structural_audit_df))

print(
    "Overall Structural Audit:",
    "PASSED" if (
        structural_audit_df["Shape Valid"].all()
        and structural_audit_df["Column Order Valid"].all()
    ) else "FAILED"
)

,Table,Actual Rows,Expected Rows,Actual Columns,Expected Columns,Column Order Valid,Shape Valid
0,DimRegion,21,21,5,5,True,True
1,DimDate,1781,1781,8,8,True,True
2,DimAgeGroup,12,12,2,2,True,True
3,DimNote,1617,1617,3,3,True,True
4,FactPopulation,210,210,5,5,True,True
5,FactNational,1781,1781,20,20,True,True
6,FactRegional,37401,37401,21,21,True,True



Valid Structures: 7 / 7
Valid Column Orders: 7 / 7
Overall Structural Audit: PASSED


### 10.3 Final Data Integrity Validation

This section validates the integrity of the final datasets before SQL Server loading.

The checks include:
- Missing primary key values.
- Duplicate primary key combinations.
- Foreign key reference validity.
- Preservation of original row counts.

The validation ensures that the final datasets are safe for the loading process.

In [71]:
# Define primary and foreign key relationships

pk_definitions = {
    "DimRegion": ["region_code"],
    "DimDate": ["date_key"],
    "DimAgeGroup": ["age_group_key"],
    "DimNote": ["note_key"],
    "FactPopulation": ["region_code", "age_group_key"],
    "FactNational": ["date_key"],
    "FactRegional": ["region_code", "date_key"]
}

integrity_results = []

for table_name, pk_columns in pk_definitions.items():

    df = final_datasets[table_name]

    missing_keys = df[pk_columns].isna().any(axis=1).sum()
    duplicate_keys = df.duplicated(subset=pk_columns).sum()

    integrity_results.append({
        "Table": table_name,
        "Missing PK Rows": missing_keys,
        "Duplicate PK Rows": duplicate_keys,
        "Integrity Valid": missing_keys == 0 and duplicate_keys == 0
    })

integrity_df = pd.DataFrame(integrity_results)

display(integrity_df)

print(
    "\nValid Tables:",
    integrity_df["Integrity Valid"].sum(),
    "/",
    len(integrity_df)
)

print(
    "Overall Integrity Audit:",
    "PASSED" if integrity_df["Integrity Valid"].all() else "FAILED"
)

,Table,Missing PK Rows,Duplicate PK Rows,Integrity Valid
0,DimRegion,0,0,True
1,DimDate,0,0,True
2,DimAgeGroup,0,0,True
3,DimNote,0,0,True
4,FactPopulation,0,0,True
5,FactNational,0,0,True
6,FactRegional,0,0,True



Valid Tables: 7 / 7
Overall Integrity Audit: PASSED


In [72]:
foreign_key_relationships = [
    ("FactPopulation", "region_code", "DimRegion", "region_code"),
    ("FactPopulation", "age_group_key", "DimAgeGroup", "age_group_key"),
    ("FactNational", "date_key", "DimDate", "date_key"),
    ("FactRegional", "region_code", "DimRegion", "region_code"),
    ("FactRegional", "date_key", "DimDate", "date_key"),
    ("DimNote", "date_key", "DimDate", "date_key")
]

fk_integrity_results = []

for child_table, child_col, parent_table, parent_col in foreign_key_relationships:

    child_df = final_datasets[child_table]
    parent_df = final_datasets[parent_table]

    parent_values = set(parent_df[parent_col].dropna())

    missing_reference_count = (
        ~child_df[child_col].isin(parent_values)
    ).sum()

    fk_integrity_results.append({
        "Child Table": child_table,
        "Child Column": child_col,
        "Parent Table": parent_table,
        "Parent Column": parent_col,
        "Missing References": missing_reference_count,
        "FK Integrity Valid": missing_reference_count == 0
    })

fk_integrity_df = pd.DataFrame(fk_integrity_results)

display(fk_integrity_df)

print(
    "\nValid Foreign Keys:",
    fk_integrity_df["FK Integrity Valid"].sum(),
    "/",
    len(fk_integrity_df)
)

print(
    "Overall Foreign Key Integrity:",
    "PASSED" if fk_integrity_df["FK Integrity Valid"].all() else "FAILED"
)

,Child Table,Child Column,Parent Table,Parent Column,Missing References,FK Integrity Valid
0,FactPopulation,region_code,DimRegion,region_code,0,True
1,FactPopulation,age_group_key,DimAgeGroup,age_group_key,0,True
2,FactNational,date_key,DimDate,date_key,0,True
3,FactRegional,region_code,DimRegion,region_code,0,True
4,FactRegional,date_key,DimDate,date_key,0,True
5,DimNote,date_key,DimDate,date_key,0,True



Valid Foreign Keys: 6 / 6
Overall Foreign Key Integrity: PASSED


### 10.5 Load Preparation

This section prepares the final datasets for loading into SQL Server.

The preparation includes:
- Defining the correct loading order.
- Verifying that all required target datasets are available.
- Calculating expected row counts.
- Preparing a load manifest for tracking the loading process.

No database connection or data insertion will be performed in this step.

In [73]:
# Define the loading order based on foreign key dependencies

load_order = [
    "DimRegion",
    "DimDate",
    "DimAgeGroup",
    "FactPopulation",
    "FactNational",
    "FactRegional",
    "DimNote"
]

# Prepare a load manifest

load_manifest = []

for table_name in load_order:

    df = final_datasets[table_name]

    load_manifest.append({
        "Table": table_name,
        "Expected Rows": len(df),
        "Expected Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Load Status": "Pending"
    })

load_manifest_df = pd.DataFrame(load_manifest)

display(load_manifest_df)

print("\nTotal Expected Rows:",
      load_manifest_df["Expected Rows"].sum())

print("Total Tables:",
      len(load_manifest_df))

print("Load Preparation:",
      "READY" if set(load_order) == set(final_datasets.keys()) else "FAILED")

,Table,Expected Rows,Expected Columns,Missing Values,Load Status
0,DimRegion,21,5,0,Pending
1,DimDate,1781,8,0,Pending
2,DimAgeGroup,12,2,0,Pending
3,FactPopulation,210,5,0,Pending
4,FactNational,1781,20,4880,Pending
5,FactRegional,37401,21,102480,Pending
6,DimNote,1617,3,14,Pending



Total Expected Rows: 42823
Total Tables: 7
Load Preparation: READY


### 10.6 Export Final Datasets

This step exports the seven validated final datasets into CSV files.

The exported files will be used by the independent SQL Server loading script.

Original datasets will remain unchanged, and missing values will be preserved.

In [74]:
from pathlib import Path

EXPORT_DIR = PROJECT_ROOT / "Src" / "ETL Output"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_file_names = {
    "DimRegion": "dim_region.csv",
    "DimDate": "dim_date.csv",
    "DimAgeGroup": "dim_age_group.csv",
    "DimNote": "dim_note.csv",
    "FactPopulation": "fact_population.csv",
    "FactNational": "fact_national.csv",
    "FactRegional": "fact_regional.csv"
}

for table_name, file_name in export_file_names.items():

    file_path = EXPORT_DIR / file_name

    final_datasets[table_name].to_csv(
        file_path,
        index=False,
        encoding="utf-8"
    )

    print(f"Exported {table_name}: {file_path}")

Exported DimRegion: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\dim_region.csv
Exported DimDate: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\dim_date.csv
Exported DimAgeGroup: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\dim_age_group.csv
Exported DimNote: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\dim_note.csv
Exported FactPopulation: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\fact_population.csv
Exported FactNational: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\fact_national.csv
Exported FactRegional: c:\Users\iamasd\Desktop\project depi\EpiShield-AI-1\Src\ETL Output\fact_regional.csv
